# MBPO — apprendre une politique en rêvant un peu, mais pas trop

> **MBPO** = *Model-Based Policy Optimization* ([Janner, Fu, Zhang, Levine, NeurIPS 2019,
> *« When to Trust Your Model »*](https://proceedings.neurips.cc/paper/2019/hash/5faf461eff3099671ad63c6f3f094f7f-Abstract.html)). MBPO récupère le meilleur des deux mondes : l'**efficacité-échantillon**
> du model-based et la **réponse instantanée** d'une politique apprise. Il atteint sur HalfCheetah le
> niveau de SAC avec une fraction des interactions réelles.

Place dans la lignée model-based : **Dyna → PILCO → PETS → MBPO → Dreamer**.

| Notebook | Méthode | Modèle | Comment on agit |
|----------|---------|--------|-----------------|
| 11 — PETS | ensembles + planning | ensemble probabiliste | **CEM/MPC en ligne** (replanifie à chaque pas) |
| **12 — MBPO** (ici) | ensembles + **politique** | ensemble probabiliste (prédit aussi $r$) | **politique SAC** apprise sur données réelles + imaginées |

L'idée tient en une phrase : **utiliser le modèle non pas pour planifier, mais pour fabriquer des
données d'entraînement bon marché** sur lesquelles on muscle une politique off-policy (SAC). Tout le
défi — et tout ce notebook — est de se servir du modèle *sans* que ses erreurs n'empoisonnent la
politique. La réponse de MBPO : des **rollouts courts**, branchés sur de vrais états.

## 1. Ce que PETS faisait payer

PETS planifie **en ligne** : à chaque pas réel, il lance une optimisation CEM qui simule des milliers
de trajectoires dans le modèle pour choisir *une* action. C'est puissant et très sample-efficient,
mais **coûteux à l'exécution** — l'agent « réfléchit » longuement à chaque pas, et il n'a aucune
politique réutilisable : éteignez le planificateur, il ne sait plus rien faire.

MBPO renverse la perspective. Plutôt que de *planifier* avec le modèle, il s'en sert comme d'un
**générateur de données** : il imagine de courtes transitions et entraîne dessus une **politique
paramétrique** (un réseau SAC). Au test, cette politique répond en un seul passage avant — instantané,
comme SAC ou PILCO. On garde donc l'efficacité-échantillon (on apprend beaucoup de chaque vraie
interaction) *et* on récupère une politique rapide.

## 2. Le marché de MBPO : des données réelles rares, des données imaginées abondantes

Interagir avec le **vrai** monde est cher (temps, usure, risque) : les transitions réelles sont **rares
et précieuses**. Interroger un **modèle** appris est quasi gratuit : on peut en générer des millions.
MBPO exploite ce déséquilibre.

À chaque petite poignée de pas réels, MBPO :
1. **affine son modèle** sur tout ce qu'il a vraiment vécu ;
2. **imagine** une multitude de courtes transitions à partir d'états réels ;
3. **entraîne intensivement** sa politique SAC sur ce mélange réel + imaginé — bien plus de mises à
   jour de gradient que ne le permettraient les seules données réelles.

Ce dernier point — beaucoup d'updates par pas réel (*update-to-data ratio* élevé) — est le vrai moteur
de l'efficacité-échantillon. Le modèle sert à **démultiplier** chaque vraie expérience.

## 3. MBPO, c'est Dyna en version moderne

Au notebook [09](./09_dyna_q_dyna_q_plus_deep_dyna_walkthrough.ipynb), **Dyna** alternait déjà : agir dans le vrai monde → ajuster un modèle → « halluciner »
des transitions pour entraîner la valeur. MBPO est exactement cette idée, *deep* et off-policy :

> **MBPO = Dyna + ensemble probabiliste (comme PETS) + SAC + rollouts courts branchés.**

```mermaid
flowchart TB
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    env["Environnement réel"]

    subgraph loop["Boucle MBPO"]
        collect["1. COLLECTER<br/>Agir avec la politique dans le vrai monde<br/>$$~(s,a,r,s') \to \mathcal{D}_{\text{env}}$$"]
        fit["2. AJUSTER<br/>Ré-entraîner l'ensemble probabiliste<br/>$$~p_\theta(s',\,r \mid s,a)$$"]
        imagine["3. IMAGINER<br/>Rollouts courts de k pas branchés sur états réels<br/>$$~\to \mathcal{D}_{\text{model}}$$"]
        train["4. ENTRAÎNER<br/>G updates SAC sur un mélange réel + imaginé<br/>$$~\rho\,\mathcal{D}_{\text{env}} \cup (1-\rho)\,\mathcal{D}_{\text{model}}$$"]
    end

    env -->|"politique courante"| collect
    collect -->|"transitions exactes"| fit
    fit -->|"modèle du monde"| imagine
    imagine -->|"transitions imaginées"| train
    train -->|"politique améliorée $$~\pi$$"| env
```


| Quick-fact | MBPO |
|---|---|
| Type | model-based, **apprend une politique** (pas de planning en ligne) |
| Modèle | ensemble probabiliste prédisant $[\Delta s,\ r]$ |
| Politique | SAC (off-policy, entropie max) |
| Données | réel (D_env) **+** imaginé (D_model), mélange `real_ratio` |
| Levier d'efficacité | rollouts courts + beaucoup d'updates par pas réel |
| Environnement (ici) | HalfCheetah-v5 (obs 17-dim, action 6-dim $\in [−1,1]$) |

In [ ]:
import collections
import copy
import math
import time
from pathlib import Path
from tqdm.auto import tqdm
from IPython.display import Video, display

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.distributions as dist_module
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float32)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

print("Environnement prêt — torch", torch.__version__)


---
# Partie I — Les deux briques héritées (rappels)

## I.1 — L'ensemble probabiliste (rappel express → notebook [11](./10b_deep_pilco_walkthrough.ipynb))

On réutilise tel quel le modèle du monde de PETS : un **ensemble de $B$ réseaux probabilistes**, chacun
prédisant une gaussienne sur la dynamique, entraîné par **vraisemblance gaussienne négative**. Deux
incertitudes y cohabitent : l'**aléatoire** (têtes de variance de chaque membre) et l'**épistémique**
(désaccord entre membres). Pour MBPO, l'épistémique sert de garde-fou implicite : là où les membres
divergent, les rollouts imaginés sont peu fiables.

*(Toute la construction — NLL, bornes de logvar, bootstrap, décomposition aléatoire/épistémique — est
détaillée pas à pas dans le notebook 11. On la considère acquise ici.)*

## I.2 — Nouveauté vs PETS : le modèle prédit aussi la récompense

PETS supposait la **récompense connue** (on lui donnait la formule de HalfCheetah). MBPO lève cette
hypothèse : le modèle prédit **à la fois** la variation d'état **et** la récompense. Chaque membre sort
donc un vecteur augmenté :

$$f_\theta(s, a) \;\approx\; \big[\ \underbrace{\Delta s}_{\in\,\mathbb{R}^{17}},\ \underbrace{r}_{\in\,\mathbb{R}}\ \big]
\quad\Longrightarrow\quad s' = s + \Delta s,\quad r = r_\theta(s,a)$$

| Terme | Rôle |
|-------|------|
| $\Delta s \in \mathbb{R}^{17}$ | variation d'état (comme PETS) |
| $r \in \mathbb{R}$ | **récompense prédite** : on n'a plus besoin de la connaître |

**Bénéfice caché.** Comme on n'a plus besoin de reconstruire la position $x$ pour calculer la
récompense, on peut garder l'**observation HalfCheetah standard (17-dim, sans $x$)**. C'est important :
la position $x$ croît sans cesse quand le guépard avance — une feature *non-stationnaire* qui perturbe
une politique. En la laissant de côté, SAC apprend sur des entrées de statistique stable. La vitesse
avant, elle, est déjà dans l'observation, donc la tête de récompense l'apprend sans peine.

## I.3 — SAC en accéléré (le détail est au notebook [08](./08_sac_halfcheetah_walkthrough.ipynb))

La politique de MBPO est **SAC** (*Soft Actor-Critic*), l'algorithme off-policy à entropie maximale.
On en redonne juste les quatre idées, le temps de fixer le vocabulaire ; le notebook 08 le construit
brique par brique.

- **Objectif entropique** : SAC maximise la récompense *et* l'entropie de la politique
  $\mathcal{H}(\pi(\cdot|s))$ — agir bien **tout en restant aussi imprévisible que possible**, ce qui
  maintient l'exploration et la robustesse.
- **Twin critics** : deux réseaux $Q$ dont on prend le **minimum** pour contrer la surestimation.
- **Température $\alpha$ auto-réglée** : le poids de l'entropie s'ajuste seul pour viser une entropie
  cible (heuristique $-\dim(\text{action})$).
- **Politique squashed-gaussian** : on échantillonne une gaussienne puis on la passe dans un $\tanh$
  pour borner l'action, avec une correction du log-prob.

Pour MBPO, l'essentiel est que SAC est off-policy : il apprend de transitions stockées dans un buffer, quelle que soit leur origine — réelles ou imaginées. C'est exactement ce qu'il faut pour se nourrir des rollouts du modèle. Le choix de SAC n'a d'ailleurs rien d'obligatoire : on le retient ici parce qu'il est standard et parmi les plus performants sur le contrôle continu, mais MBPO est agnostique à la politique — n'importe quel apprenant off-policy (TD3, DDPG, …) peut prendre sa place, du moment qu'il sait apprendre depuis un buffer mixte réel + imaginé.

## I.4 — L'objectif entropique, en une équation

$$J(\pi) = \sum_t \mathbb{E}_{(s_t,a_t)\sim\pi}\Big[\, r(s_t,a_t) \;+\; \alpha\,\mathcal{H}\big(\pi(\cdot|s_t)\big) \,\Big]$$

| Terme | Rôle |
|-------|------|
| $r(s_t,a_t)$ | la récompense usuelle |
| $\alpha\,\mathcal{H}(\pi(\cdot\|s_t))$ | **bonus d'entropie** : récompense le fait de garder de l'incertitude dans le choix d'action |
| $\alpha$ | température, réglée automatiquement |

C'est tout ce dont on a besoin ici. La cible critique, la correction $\tanh$ du log-prob, la descente
sur $\alpha$ — tout est au **notebook 08**. On va réimplémenter SAC de façon compacte (Partie IV) sans
le re-commenter ligne à ligne.

Les deux briques (ensemble, SAC) sont héritées. La question neuve — celle qui fait *tout* MBPO — est :
**comment se servir d'un modèle imparfait pour générer des données sans empoisonner la politique ?**
Réponse en Partie II : des rollouts **courts**.

---
# Partie II — Rollouts courts : *when to trust your model*

## II.1 — L'erreur d'un modèle se *compose*

Un modèle appris n'est jamais parfait : à chaque pas qu'on simule, il commet une petite erreur. Le
piège, c'est que ces erreurs **s'accumulent**. Au pas 2, on part déjà d'un état légèrement faux, donc
on s'éloigne encore ; au pas 3, davantage ; et ainsi de suite. L'écart entre la trajectoire imaginée et
la réalité **croît avec l'horizon**, souvent de façon explosive.

C'est exactement ce qu'on a vu en fin de notebook 11 (« imagination vs réalité ») : les premiers pas
collent, puis ça diverge. Dérouler le modèle sur 100 pas pour entraîner une politique reviendrait à lui
apprendre à se comporter dans un monde **de plus en plus fictif**. La politique optimiserait des
hallucinations.

## II.2 — Pourquoi l'horizon est le bon levier

Notons $\eta[\pi]$ le **retour espéré** d'une politique $\pi$ — la somme des récompenses qu'elle accumule. MBPO ne peut pas optimiser directement $\eta_{\text{réel}}[\pi]$ (cela coûterait des interactions réelles) : il optimise $\eta_{\text{modèle}}[\pi]$, le retour **dans le modèle**, qui est quasi gratuit. Tout le pari repose donc sur une seule question : **de combien le retour-modèle peut-il s'écarter du retour-réel ?** Si l'écart est petit, améliorer la politique dans le modèle l'améliore aussi pour de vrai ; s'il est grand, on optimise une fiction. L'analyse de MBPO **borne** précisément cet écart.

Si l'erreur de généralisation du modèle est majorée par $\epsilon_m$ **par pas**, l'écart se majore schématiquement par :

$$\underbrace{\big|\,\eta_{\text{réel}}[\pi] - \eta_{\text{modèle}}[\pi]\,\big|}_{\text{écart de performance}}
\;\lesssim\; \underbrace{C_1\,k\,\epsilon_m}_{\text{erreur du modèle, croît avec }k}
\;+\; \underbrace{C_2\,\epsilon_\pi}_{\text{décalage de politique}}$$

Les deux termes ont des origines très différentes :

- **Erreur du modèle, $C_1\,k\,\epsilon_m$.** À chaque pas imaginé, le modèle se trompe d'un petit $\epsilon_m$ sur l'état (et la récompense) suivant. Sur un rollout de $k$ pas, ces erreurs **se télescopent** : le pas 2 part déjà d'un état faux, le pas 3 encore plus… L'écart cumulé grandit donc, au pire, **linéairement en $k$**. C'est le **coût des hallucinations** — la raison pour laquelle un long rollout finit dans un monde fictif.
- **Décalage de politique, $C_2\,\epsilon_\pi$.** Les états de départ viennent d'un buffer rempli par une politique **passée** ; à mesure que $\pi$ s'améliore, elle visite des états un peu différents de ceux stockés. Ce terme mesure ce décalage de distribution. Point clé : il **ne dépend pas de $k$**, et l'apprentissage **off-policy** (replay) est précisément fait pour le tolérer — on le garde donc petit « gratuitement ».

| Terme | Lecture |
|-------|---------|
| $\epsilon_m$ | erreur du modèle par pas (diminue quand on l'entraîne davantage) |
| $k$ | longueur du rollout imaginé |
| $C_1\,k\,\epsilon_m$ | **le coût des hallucinations** : il grandit avec $k$ — d'où l'intérêt de $k$ petit |
| $C_2\,\epsilon_\pi$ | décalage de distribution ; **constant en $k$**, off-policy l'atténue |

**Le seul terme qu'on contrôle en temps réel, c'est $k$.** $\epsilon_m$ ne baisse que lentement — il faut ré-entraîner le modèle sur davantage de données — et $\epsilon_\pi$ est déjà borné par le choix off-policy. La longueur de rollout, elle, est un **curseur immédiat** : la réduire écrase aussitôt le terme dangereux $C_1\,k\,\epsilon_m$.

Pourquoi ne pas alors prendre $k$ aussi petit que possible ? Parce que $k=0$ signifie **aucune donnée imaginée** — on retombe sur SAC pur, sans le moindre gain d'efficacité-échantillon. Il y a donc un **compromis** : plus $k$ est grand, plus on fabrique de transitions par état réel (amplification des données), mais plus elles sont biaisées. MBPO tranche du côté **prudent** — souvent $k=1$ — et n'allonge $k$ qu'à mesure que $\epsilon_m$ diminue (le *schedule* de II.4). Même à $k=1$ le gain reste énorme : chaque état réel sème une (ou quelques) transitions imaginées, et on enchaîne $G$ mises à jour de gradient dessus (l'UTD élevé de la Partie III).

C'est tout le sens du titre — *« when to trust your model »* : tant que $k\,\epsilon_m$ reste petit, le retour-modèle est un **proxy fiable** du retour-réel, donc optimiser la politique dans le modèle est **sûr** ; dès que $k$ s'allonge trop, la garantie s'évapore. *Mieux vaut beaucoup de tout petits rêves fiables qu'un long rêve qui dérape.*


In [ ]:
# Démonstration jouet de l'erreur composée dans un système 1D linéaire

np.random.seed(1)

# Vrai système : s' = 0.9*s + a + bruit N(0, 0.05)
# Modèle légèrement biaisé : s' = 0.92*s + a + bruit modèle N(0, 0.03)
TRUE_COEFF = 0.90
MODEL_COEFF = 0.92
TRUE_NOISE  = 0.05
MODEL_NOISE = 0.03

H = 30          # horizon maximal
N_SEEDS = 64    # nombre d'états de départ

# États initiaux et actions aléatoires communes
s0 = np.random.uniform(-1, 1, N_SEEDS).astype(np.float32)
actions = np.random.uniform(-0.2, 0.2, (H, N_SEEDS)).astype(np.float32)

errors_by_h = []

s_true  = s0.copy()
s_model = s0.copy()

for h in range(H):
    a = actions[h]
    # Avancement dans le vrai monde (avec bruit)
    s_true  = TRUE_COEFF  * s_true  + a + np.random.normal(0, TRUE_NOISE,  N_SEEDS).astype(np.float32)
    # Avancement dans le modèle (avec bruit modèle)
    s_model = MODEL_COEFF * s_model + a + np.random.normal(0, MODEL_NOISE, N_SEEDS).astype(np.float32)
    errors_by_h.append(np.abs(s_true - s_model))

errors_by_h = np.array(errors_by_h)   # shape (H, N_SEEDS)
mean_err  = errors_by_h.mean(axis=1)
q25 = np.percentile(errors_by_h, 25, axis=1)
q75 = np.percentile(errors_by_h, 75, axis=1)

horizons = np.arange(1, H + 1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(horizons, mean_err, lw=2, label="Erreur moyenne")
ax.fill_between(horizons, q25, q75, alpha=0.3, label="IQR (25–75 %)")
ax.set_xlabel("Horizon de rollout $k$")
ax.set_ylabel("|s_modèle − s_vrai|")
ax.set_title("Erreur composée : écart modèle / réalité en fonction de l'horizon")
ax.legend()
plt.tight_layout()
plt.show()

**Lecture.** L'erreur est minuscule au premier pas puis **enfle régulièrement** avec l'horizon : c'est
la signature de l'erreur composée. La zone fiable — là où imaginer reste proche du réel — est un
**court** segment au départ. MBPO ne dépasse pas cette zone : il branche des rollouts de quelques pas,
puis repart d'un *vrai* état. On ne fait jamais confiance au modèle au-delà de son horizon de fiabilité.

## II.3 — La parade : brancher de courts rollouts sur de *vrais* états

Plutôt qu'un long rollout depuis l'état initial, MBPO fait des **rollouts branchés** (*branched
rollouts*) : on **tire un état réel** déjà visité (depuis $D_{\text{env}}$), et on n'imagine que $k$ pas
à partir de lui. Chaque court rollout démarre donc d'un point **ancré dans la réalité**, jamais d'une
accumulation d'hallucinations.

**Analogie du GPS.** Suivre un itinéraire calculé une fois pour toutes sur une carte imparfaite, c'est
risquer de finir dans un champ. MBPO fait comme un conducteur prudent : il regarde la carte sur **les
quelques prochains carrefours seulement**, avance pour de vrai, puis **recalcule depuis sa position
réelle**. La carte (le modèle) n'est sollicitée que là où elle reste fiable.

## II.4 — Allonger $k$ à mesure que le modèle s'améliore

Au début, le modèle est mauvais ($\epsilon_m$ grand) : on reste à $k=1$. Plus on collecte de données
réelles, meilleur devient le modèle, et on peut **se permettre des rollouts un peu plus longs** sans
trop d'erreur. MBPO utilise donc souvent un **schedule linéaire** de la longueur de rollout :

$$k(\text{époque}) = \min\!\Big(k_{\max},\ k_{\min} + \big\lfloor \tfrac{\text{époque} - e_0}{e_1 - e_0}\,(k_{\max}-k_{\min}) \big\rfloor\Big)$$

C'est un curseur sur le compromis « **quantité de données imaginées** vs **fiabilité** » : on n'achète
des rollouts plus longs qu'une fois le modèle digne de confiance. Dans notre démo courte, on restera à
$k$ petit (1 à quelques pas).

In [ ]:
# Schedule linéaire de la longueur de rollout k(epoch)

k_min, k_max = 1, 5
e0, e1 = 0, 15
n_epochs = 20

epochs = np.arange(n_epochs)

def k_schedule(epoch, k_min, k_max, e0, e1):
    # Longueur de rollout croissant linéairement entre k_min et k_max
    if epoch < e0:
        return k_min
    if epoch > e1:
        return k_max
    progress = (epoch - e0) / max(e1 - e0, 1)
    return min(k_max, k_min + int(progress * (k_max - k_min)))

k_values = [k_schedule(e, k_min, k_max, e0, e1) for e in epochs]

fig, ax = plt.subplots(figsize=(9, 4))
ax.step(epochs, k_values, where="post", lw=2, color="steelblue")
ax.set_xlabel("Époque")
ax.set_ylabel("Longueur de rollout $k$")
ax.set_title("Schedule linéaire de $k$ : prudent au début, ambitieux ensuite")
ax.set_yticks(range(k_min, k_max + 1))
plt.tight_layout()
plt.show()

**Lecture.** La longueur de rollout monte par paliers : prudence au début (modèle naïf), ambition
ensuite (modèle mûr). C'est le « *when to trust your model* » du titre — on n'étend la confiance au
modèle qu'au rythme où il la mérite. Reste à organiser le flux de données : un réservoir pour le réel,
un pour l'imaginé. C'est la Partie III.

# Partie III — Deux réservoirs de données

## III.1 — Réel vs imaginé : $D_{\text{env}}$ et $D_{\text{model}}$

MBPO sépare ses données en **deux réservoirs**, parce qu'elles n'ont ni la même valeur ni la même durée de vie :

- **$D_{\text{env}}$ — le réel.** Transitions issues du vrai environnement : **rares, chères, mais exactes**. Double rôle : (a) elles **entraînent le modèle**, et (b) elles fournissent les **états de départ** des rollouts (on ne rêve jamais qu'à partir d'un point réellement visité). Elles restent **toujours valides** — une vraie transition d'il y a longtemps reste vraie.
- **$D_{\text{model}}$ — l'imaginé.** Transitions générées par les rollouts branchés : **abondantes, quasi gratuites, mais approchées**. Elles forment l'essentiel du batch qui **muscle la politique**.

Pourquoi ne pas tout fondre dans un seul buffer ? Deux raisons :

1. **Doser** — les garder distincts permet de choisir, à chaque batch, **quelle proportion** de réel et d'imaginé on mélange (le `real_ratio` de III.2). Dans un pool unique, ce contrôle est perdu.
2. **Rafraîchir** — $D_{\text{model}}$ est **non-stationnaire** : les transitions imaginées il y a longtemps viennent d'un modèle **périmé**. On lui donne donc une **capacité limitée** et on jette les vieilles imaginations au profit des fraîches. $D_{\text{env}}$, lui, n'a pas ce problème et peut tout conserver.

## III.2 — Le dosage `real_ratio` : une pincée de réel ancre la politique

À chaque mise à jour de SAC, on échantillonne un **batch mixte** : une fraction `real_ratio` tirée de $D_{\text{env}}$, le reste de $D_{\text{model}}$.

$$\text{batch} = \underbrace{\rho\,|B|}_{\text{réel}} \ \cup\ \underbrace{(1-\rho)\,|B|}_{\text{imaginé}},
\qquad \rho = \texttt{real\_ratio}\ (\approx 0.05\text{–}0.1)$$

L'idée est celle d'un **ancrage**. L'imaginé apporte la **quantité** — c'est lui qui démultiplie les données — mais il est entaché des **biais du modèle** ; entraînée uniquement dessus, la politique apprendrait à **exploiter ces biais** (« hallucinations rentables ») et dériverait loin du réel. Glisser ne serait-ce que **5 % de transitions vraies** dans chaque batch agit comme un **rappel à l'ordre** : à chaque update, un petit signal exact recale la politique sur le monde réel.

Pourquoi si **peu** ? Parce que les deux excès se paient :

- **trop de réel** → on retombe vers SAC pur, on perd l'amplification du modèle, l'efficacité-échantillon s'effondre ;
- **trop peu de réel** → plus rien n'ancre la politique, qui part dans les hallucinations.

Comme $k$, `real_ratio` est donc un **curseur de confiance** : plus on fait confiance au modèle, plus on peut baisser la dose de réel.

## III.3 — Le vrai moteur : beaucoup d'updates par pas réel

Voici le levier d'efficacité-échantillon de MBPO, souvent sous-estimé. Un agent off-policy classique (SAC) fait **environ une** mise à jour de gradient par pas réel : il a beau réutiliser son buffer, son rythme reste **plafonné par le débit de vraies données** — pas de nouvelle interaction, pas de nouvelle leçon. MBPO casse ce plafond : comme le modèle **fabrique** des transitions à volonté, on peut faire **$G$ mises à jour** (20, 40…) par pas réel.

$$\text{UTD (update-to-data)} = \frac{\text{nombre d'updates de gradient}}{\text{nombre de pas réels}} \;=\; G \;\gg\; 1$$

Chaque vraie interaction est ainsi **pressée comme un citron** : on en extrait des dizaines de leçons au lieu d'une. C'est *là* que naît le « 10× moins d'interactions réelles » — pas dans le modèle en soi, mais dans le **nombre d'updates** qu'il autorise.

Attention toutefois : un UTD élevé **amplifie ce qu'on lui donne**. Si le modèle est encore grossier, marteler la politique avec $G$ updates par pas ne fait que **graver ses biais** plus vite (c'est exactement ce qu'on observera sur HalfCheetah). L'UTD élevé n'est donc puissant **qu'accompagné** des garde-fous des parties précédentes : rollouts **courts** ($k$ petit) et **ensemble** (incertitude). Le modèle reste un moyen — le but est de **muscler la politique bien plus vite** que ne le permet le monde réel seul.


In [ ]:
# Illustration de la composition du batch mixte pour deux valeurs de real_ratio

def sample_mixed_toy(n_real, n_model, batch_size, real_ratio):
    # Retourne le nombre d'échantillons réels et imaginés dans le batch
    real_count  = min(int(round(real_ratio * batch_size)), n_real)
    model_count = min(batch_size - real_count, n_model)
    # Compenser si un côté est vide
    if real_count == 0:
        model_count = min(batch_size, n_model)
    if model_count == 0:
        real_count = min(batch_size, n_real)
    return real_count, model_count

N_REAL  = 1000
N_MODEL = 50000
BATCH   = 256
ratios  = [0.05, 0.5]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, ratio in zip(axes, ratios):
    rc, mc = sample_mixed_toy(N_REAL, N_MODEL, BATCH, ratio)
    ax.bar(["Réel", "Imaginé"], [rc, mc], color=["steelblue", "coral"], edgecolor="black")
    ax.set_title(f"real_ratio = {ratio}")
    ax.set_ylabel("Nombre de transitions dans le batch")
    for bar, val in zip(ax.patches, [rc, mc]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f"{val}", ha="center", va="bottom", fontsize=11)
plt.suptitle("Composition du batch mixte (batch=256)", fontsize=12)
plt.tight_layout()
plt.show()

**Lecture.** À `real_ratio = 0.05`, le batch d'entraînement de la politique est rempli à 95 %
d'imaginé : le modèle porte la quantité, le réel ne fait qu'ancrer. C'est ce mélange, répété $G$ fois
par pas réel, qui transforme une poignée d'interactions en un entraînement dense. On a maintenant toutes
les pièces conceptuelles — passons au code.

# Partie IV — Tout assembler, from scratch

On a maintenant **toutes les idées** : modèle probabiliste (ensemble + tête de récompense), rollouts courts branchés, deux buffers, dosage `real_ratio`, UTD élevé. Il reste à les **souder** en un agent qui tourne. Le code des cellules suivantes est volontairement **dense et peu commenté** (la pédagogie ligne-à-ligne est dans les notebooks 08 et 11) ; cette cellule est donc la **carte** : elle nomme chaque brique et montre **comment tout s'emboîte**.

Les briques, et où elles vivent dans le code :

| Brique | Rôle | Objet dans le code |
|--------|------|--------------------|
| Modèle du monde | prédit $[\Delta s,\ r]$, avec incertitude | `ProbabilisticEnsemble` |
| Politique | agit *et* s'entraîne (off-policy) | `SacLearner` |
| Buffer réel | transitions exactes $\mathcal{D}_{\text{env}}$ | `ReplayBuffer` |
| Buffer imaginé | transitions du modèle $\mathcal{D}_{\text{model}}$ | `ModelBuffer` |
| Mélange | compose le batch $\rho$ réel + $(1-\rho)$ imaginé | `sample_mixed` |
| Rollouts branchés | $k$ pas imaginés depuis des états réels | `generate_rollouts` |
| Chef d'orchestre | enchaîne collecte → fit → imagine → updates | `run_mbpo` |

```mermaid
flowchart TB
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    pol["Politique SAC<br/>acteur + 2 critiques + température<br/>$$~\pi_\phi(a \mid s)$$"]
    env["Environnement réel"]

    subgraph model["Modèle du monde"]
        ens["Ensemble probabiliste<br/>$$~p_\theta(s', r \mid s, a)$$"]
        roll["generate_rollouts<br/>k pas branchés sur états réels"]
    end

    denv["Buffer réel<br/>$$~\mathcal{D}_{\text{env}}$$"]
    dmodel["Buffer imaginé<br/>$$~\mathcal{D}_{\text{model}}$$"]
    mix["sample_mixed<br/>batch réel + imaginé"]
    upd["update SAC<br/>critic + actor + alpha"]

    pol -->|"agit dans le vrai monde"| env
    env -->|"transitions exactes"| denv
    denv -->|"entraîne le modèle"| ens
    denv -->|"états de départ"| roll
    ens -->|"propage la dynamique"| roll
    pol -->|"actions imaginées"| roll
    roll -->|"transitions imaginées"| dmodel
    denv -->|"fraction $$~\rho$$"| mix
    dmodel -->|"fraction $$~1-\rho$$"| mix
    mix -->|"G updates / pas réel"| upd
    upd -->|"améliore"| pol
```

## IV.1 — L'ensemble probabiliste (avec tête de récompense)

On reprend l'ensemble du notebook 11 — $B$ réseaux probabilistes, chacun sortant une **gaussienne** et entraîné par bootstrap — avec **une sortie de plus** : la récompense. Chaque membre prédit donc une gaussienne sur $[\Delta s,\ r]$, de dimension $\text{obs\_dim}+1$ (ici $17+1=18$). Le reste est **identique** au nb 11 : têtes moyenne / log-variance, bornes douces sur la log-variance, NLL gaussienne, normalisation entrée/sortie, bootstrap. La **récompense prédite** est ce qui autorise des transitions *complètes* $(s,a,r,s')$ entièrement imaginées, sans connaître la formule de récompense de l'environnement (cf. I.2).

## IV.2 — SAC compact : le rappel pour lire le code

`SacLearner` condense tout SAC en une classe. Voici ses **quatre pièces** et l'**anatomie d'`update()`**, pour parcourir le code sans s'y perdre (le détail ligne-à-ligne est au notebook 08).

**Les quatre pièces.**
- **Acteur squashed-gaussian** : il sort une moyenne et un écart-type, échantillonne $u\sim\mathcal{N}$, puis **borne** l'action par $a=\tanh(u)$ (avec correction du log-prob) — d'où des actions toujours dans $[-1,1]$.
- **Twin critics** : deux réseaux $Q_1, Q_2$ dont on prend le **minimum**, parade anti-surestimation. La cible $y$ n'est pas calculée avec ces réseaux mais avec une **copie figée** ($Q_{\phi'}$), pour ne pas poursuivre une cible mouvante. Cette copie suit lentement par **Polyak**, $\phi' \leftarrow \tau\,\phi + (1-\tau)\,\phi'$ avec $\tau$ petit ($\approx 0.005$) : une référence stable pendant que les critiques apprennent vite.
- **Température $\alpha$ auto-réglée** : ajustée seule pour viser une entropie cible $-\dim(a)$.
- **Objectif entropique** : la récompense est augmentée d'un bonus $\alpha\,\mathcal{H}(\pi)$ — agir bien *en restant imprévisible*.

**L'anatomie d'`update(batch)`** — un batch = mélange réel + imaginé :
1. **Cible critique** : $y = r + \gamma\,(1-\text{done})\,\big(\min_i Q_{\phi'_i}(s',a') - \alpha\log\pi(a'\mid s')\big)$, puis MSE sur $Q_1$ et $Q_2$.
2. **Acteur** : minimiser $\big(\alpha\log\pi(a\mid s) - \min_i Q_i(s,a)\big)$ — viser les actions à fort $Q$ tout en gardant de l'entropie.
3. **Température** : une descente sur $\alpha$ vers l'entropie cible.
4. **Polyak** : $\phi' \leftarrow \tau\,\phi + (1-\tau)\,\phi'$.

Le détail qui compte pour MBPO : `update()` prend un **batch explicite** (et non un buffer interne), précisément pour qu'on puisse lui servir **n'importe quel mélange** réel + imaginé via `sample_mixed`.


In [ ]:
# ============================================================
# Ensemble probabiliste from-scratch (avec tête de récompense)
# ============================================================

def gaussian_nll(mean, logvar, target):
    # NLL diagonale-gaussienne
    inv_var = torch.exp(-logvar)
    per_sample = 0.5 * ((target - mean) ** 2 * inv_var + logvar).sum(dim=-1)
    return per_sample.mean()


class ProbabilisticMLP(nn.Module):
    # Réseau probabiliste : prédit (mean, logvar) pour [Δs, r]
    def __init__(self, input_dim, output_dim, hidden_dim=200, n_layers=4):
        super().__init__()
        self.output_dim = output_dim
        layers = []
        in_f = input_dim
        for _ in range(n_layers):
            layers += [nn.Linear(in_f, hidden_dim), nn.SiLU()]
            in_f = hidden_dim
        layers.append(nn.Linear(in_f, 2 * output_dim))
        self.net = nn.Sequential(*layers)
        # Bornes douces sur logvar (apprenables)
        self.max_logvar = nn.Parameter(torch.full((output_dim,), 0.5))
        self.min_logvar = nn.Parameter(torch.full((output_dim,), -10.0))

    def forward(self, x):
        out = self.net(x)
        raw_mean, raw_logvar = out.chunk(2, dim=-1)
        logvar = self.max_logvar - F.softplus(self.max_logvar - raw_logvar)
        logvar = self.min_logvar + F.softplus(logvar - self.min_logvar)
        return raw_mean, logvar


In [ ]:
class ProbabilisticEnsemble(nn.Module):
    # Ensemble de B membres, output_dim = obs_dim + 1
    def __init__(self, input_dim, output_dim, ensemble_size=7, hidden_dim=200, n_layers=4):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.ensemble_size = ensemble_size
        self.members = nn.ModuleList([
            ProbabilisticMLP(input_dim, output_dim, hidden_dim, n_layers)
            for _ in range(ensemble_size)
        ])
        # Buffers de normalisation
        self.register_buffer("x_mean", torch.zeros(input_dim))
        self.register_buffer("x_std",  torch.ones(input_dim))
        self.register_buffer("y_mean", torch.zeros(output_dim))
        self.register_buffer("y_std",  torch.ones(output_dim))

    def fit_normalizer(self, X, Y):
        self.x_mean.copy_(X.mean(0))
        self.x_std.copy_(X.std(0).clamp_min(1e-6))
        self.y_mean.copy_(Y.mean(0))
        self.y_std.copy_(Y.std(0).clamp_min(1e-6))

    def _nx(self, X): return (X - self.x_mean) / self.x_std
    def _ny(self, Y): return (Y - self.y_mean) / self.y_std
    def _dy(self, Y): return Y * self.y_std + self.y_mean

    def fit(self, X, Y, steps=200, batch_size=256, lr=1e-3, weight_decay=1e-5):
        # Entraîne tous les membres par bootstrap (NLL + régularisation logvar)
        self.fit_normalizer(X, Y)
        Xn, Yn = self._nx(X), self._ny(Y)
        N = Xn.shape[0]
        nlls = []
        for member in self.members:
            member.train()
            opt = torch.optim.Adam(member.parameters(), lr=lr, weight_decay=weight_decay)
            idx = torch.randint(0, N, (N,))
            Xb, Yb = Xn[idx], Yn[idx]
            last_nll = 0.0
            for _ in range(steps):
                bi = torch.randint(0, N, (min(batch_size, N),))
                opt.zero_grad()
                mn, lv = member(Xb[bi])
                loss = gaussian_nll(mn, lv, Yb[bi]) + 0.01 * (member.max_logvar.sum() - member.min_logvar.sum())
                loss.backward()
                opt.step()
                last_nll = float(gaussian_nll(mn, lv, Yb[bi]).item())
            member.eval()
            nlls.append(last_nll)
        return float(np.mean(nlls))

    @torch.no_grad()
    def predict(self, x):
        # Retourne (means, vars) shapes [B, E, output_dim]
        xn = self._nx(x)
        all_m, all_v = [], []
        for member in self.members:
            mn, lv = member(xn)
            all_m.append(self._dy(mn))
            all_v.append(torch.exp(lv) * self.y_std ** 2)
        return torch.stack(all_m, 1), torch.stack(all_v, 1)

    @torch.no_grad()
    def propagate(self, states, actions, model_idx):
        # Un pas : retourne (next_states [B, obs_dim], rewards [B])
        B = states.shape[0]
        obs_dim = states.shape[1]
        x = torch.cat([states, actions], dim=-1)
        xn = self._nx(x)
        means_n = torch.zeros(B, self.output_dim)
        logvs_n = torch.zeros(B, self.output_dim)
        for m_idx, member in enumerate(self.members):
            mask = (model_idx == m_idx)
            if not mask.any():
                continue
            member.eval()
            mn, lv = member(xn[mask])
            means_n[mask] = mn
            logvs_n[mask] = lv
        eps = torch.randn_like(means_n)
        sample_n = means_n + torch.exp(0.5 * logvs_n) * eps
        sample = self._dy(sample_n)
        delta = sample[:, :obs_dim]
        reward = sample[:, obs_dim]
        return states + delta, reward

    @torch.no_grad()
    def disagreement(self, X):
        means, _ = self.predict(X)
        return float(means.std(dim=1).mean().item())


print("✓ ProbabilisticEnsemble défini")


In [ ]:
# Mini-test : ensemble jouet (Δs = -0.1*s, r = vitesse-like)
torch.manual_seed(42)
np.random.seed(42)

OBS_D, ACT_D = 3, 2
B_SIZE = 5

ens = ProbabilisticEnsemble(input_dim=OBS_D + ACT_D, output_dim=OBS_D + 1,
                             ensemble_size=B_SIZE, hidden_dim=64, n_layers=2)

N = 300
s_toy = torch.randn(N, OBS_D)
a_toy = torch.randn(N, ACT_D)
# Cible : Δs = -0.1*s, r = 0.5 * mean(s)
delta_toy = -0.1 * s_toy
r_toy = 0.5 * s_toy.mean(dim=1, keepdim=True)
Y_toy = torch.cat([delta_toy, r_toy], dim=1)
X_toy = torch.cat([s_toy, a_toy], dim=1)

nll_before = ens.fit(X_toy, Y_toy, steps=5, batch_size=64)

# Entraîner davantage
nll_after = ens.fit(X_toy, Y_toy, steps=200, batch_size=64)
assert nll_after < nll_before, f"NLL devrait baisser avec plus d'entraînement : {nll_before:.3f} -> {nll_after:.3f}"

# Test shapes de predict
means, vars_ = ens.predict(X_toy[:8])
assert means.shape == (8, B_SIZE, OBS_D + 1), f"Shape predict means: {means.shape}"
assert vars_.shape  == (8, B_SIZE, OBS_D + 1)

# Test propagate
idx = torch.randint(0, B_SIZE, (8,))
ns, rw = ens.propagate(s_toy[:8], a_toy[:8], idx)
assert ns.shape == (8, OBS_D),  f"Shape next_states: {ns.shape}"
assert rw.shape == (8,),        f"Shape rewards: {rw.shape}"

# Corrélation récompense prédite vs vraie
r_pred = []
for i in range(0, N, 32):
    x_b = X_toy[i:i+32]
    idx_b = torch.zeros(x_b.shape[0], dtype=torch.long)
    _, rw_b = ens.propagate(s_toy[i:i+32], a_toy[i:i+32], idx_b)
    r_pred.append(rw_b.numpy())
r_pred = np.concatenate(r_pred)
r_true = r_toy.squeeze().numpy()
corr = float(np.corrcoef(r_pred, r_true)[0, 1])
assert corr > 0.3, f"Corrélation récompense trop faible : {corr:.3f}"

print(f"✓ predict shapes OK | propagate shapes OK | corr(r_pred, r_true)={corr:.3f}")

## IV.2 — SAC compact (implémentation condensée → détails au notebook 08)

On réimplémente SAC de façon **dense et peu commentée** : acteur squashed-gaussian (gaussienne + $\tanh$
+ correction du log-prob), twin critics avec cible Polyak, température $\alpha$ auto-réglée. Une seule
différence d'ingénierie par rapport au notebook 08 : la méthode `update` prend un **batch explicite**
(et non un buffer interne), pour qu'on puisse lui passer notre mélange réel + imaginé. Si une ligne vous
intrigue, le notebook 08 l'explique en détail.

In [ ]:
# ============================================================
# SAC compact from-scratch (acteur squashed-gaussian + twin critics + alpha auto)
# ============================================================

def _orth(layer, gain=1.0):
    nn.init.orthogonal_(layer.weight, gain=gain)
    nn.init.constant_(layer.bias, 0.0)
    return layer

class ContinuousQNetwork(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            _orth(nn.Linear(obs_dim + act_dim, hidden_dim), gain=math.sqrt(2)), nn.ReLU(),
            _orth(nn.Linear(hidden_dim, hidden_dim), gain=math.sqrt(2)), nn.ReLU(),
            _orth(nn.Linear(hidden_dim, 1), gain=1.0),
        )
    def forward(self, obs, act):
        return self.net(torch.cat([obs, act], dim=-1)).squeeze(-1)

class TwinQNetwork(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim=256):
        super().__init__()
        self.q1 = ContinuousQNetwork(obs_dim, act_dim, hidden_dim)
        self.q2 = ContinuousQNetwork(obs_dim, act_dim, hidden_dim)
    def forward(self, obs, act):
        return self.q1(obs, act), self.q2(obs, act)

class SquashedGaussianActor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim=256,
                 log_std_min=-20.0, log_std_max=2.0):
        super().__init__()
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max
        self.trunk = nn.Sequential(
            _orth(nn.Linear(obs_dim, hidden_dim), gain=math.sqrt(2)), nn.ReLU(),
            _orth(nn.Linear(hidden_dim, hidden_dim), gain=math.sqrt(2)), nn.ReLU(),
        )
        self.mean_head    = _orth(nn.Linear(hidden_dim, act_dim), gain=0.01)
        self.log_std_head = _orth(nn.Linear(hidden_dim, act_dim), gain=0.01)

    def forward(self, obs):
        f = self.trunk(obs)
        return self.mean_head(f), self.log_std_head(f).clamp(self.log_std_min, self.log_std_max)

    def sample(self, obs):
        # Retourne (action, log_prob, action_deterministe)
        mean, log_std = self.forward(obs)
        std = log_std.exp()
        u = dist_module.Normal(mean, std).rsample()
        raw = torch.tanh(u)
        log_prob = (dist_module.Normal(mean, std).log_prob(u)
                    - torch.log(1.0 - raw.pow(2) + 1e-6)).sum(dim=-1)
        return raw, log_prob, torch.tanh(mean)


In [ ]:
class SacLearner:
    # SAC opère sur des batches explicites (pas de buffer interne)
    def __init__(self, obs_dim, act_dim, hidden_dim=256,
                 gamma=0.99, tau=0.005, alpha=0.2,
                 actor_lr=3e-4, critic_lr=3e-4, alpha_lr=3e-4):
        self.gamma = gamma
        self.tau   = tau
        self.target_entropy = float(-act_dim)

        self.actor  = SquashedGaussianActor(obs_dim, act_dim, hidden_dim)
        self.critic = TwinQNetwork(obs_dim, act_dim, hidden_dim)
        self.critic_target = copy.deepcopy(self.critic)
        self.critic_target.eval()
        for p in self.critic_target.parameters():
            p.requires_grad_(False)

        self.actor_opt  = torch.optim.Adam(self.actor.parameters(),  lr=actor_lr)
        self.critic_opt = torch.optim.Adam(self.critic.parameters(), lr=critic_lr)

        self.log_alpha  = nn.Parameter(torch.log(torch.tensor(alpha)))
        self.alpha_opt  = torch.optim.Adam([self.log_alpha], lr=alpha_lr)

    @property
    def alpha(self):
        return float(self.log_alpha.exp().item())

    def select_action(self, obs, deterministic=False):
        obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            mean, _ = self.actor(obs_t)
            if deterministic:
                return torch.tanh(mean).squeeze(0).numpy().astype(np.float32)
            act, _, _ = self.actor.sample(obs_t)
        return act.squeeze(0).numpy().astype(np.float32)

    def update(self, obs, act, rew, next_obs, done):
        # ── Critic ──────────────────────────────────────────────────
        alpha_val = self.alpha
        with torch.no_grad():
            na, nlp, _ = self.actor.sample(next_obs)
            q1t, q2t = self.critic_target(next_obs, na)
            tgt = rew + self.gamma * (1.0 - done) * (torch.min(q1t, q2t) - alpha_val * nlp)
        q1, q2 = self.critic(obs, act)
        c_loss = F.mse_loss(q1, tgt) + F.mse_loss(q2, tgt)
        self.critic_opt.zero_grad(); c_loss.backward(); self.critic_opt.step()

        # ── Actor ────────────────────────────────────────────────────
        a_pi, lp, _ = self.actor.sample(obs)
        q1p, q2p = self.critic(obs, a_pi)
        a_loss = (alpha_val * lp - torch.min(q1p, q2p)).mean()
        self.actor_opt.zero_grad(); a_loss.backward(); self.actor_opt.step()

        # ── Alpha ────────────────────────────────────────────────────
        al_loss = -(self.log_alpha * (lp.detach() + self.target_entropy)).mean()
        self.alpha_opt.zero_grad(); al_loss.backward(); self.alpha_opt.step()

        # ── Polyak ───────────────────────────────────────────────────
        for p_on, p_tg in zip(self.critic.parameters(), self.critic_target.parameters()):
            p_tg.data.copy_(self.tau * p_on.data + (1.0 - self.tau) * p_tg.data)

        return {
            "critic_loss": float(c_loss.item()),
            "actor_loss":  float(a_loss.item()),
            "alpha":       self.alpha,
            "entropy":     float(-lp.mean().item()),
        }

print("✓ SacLearner défini")


In [ ]:
# Mini-test : vérifier que SacLearner s'entraîne et que select_action fonctionne
torch.manual_seed(7)
np.random.seed(7)

OBS_DIM, ACT_DIM = 17, 6
sac_test = SacLearner(OBS_DIM, ACT_DIM, hidden_dim=64)

# Batch fixe aléatoire
obs_b    = torch.randn(64, OBS_DIM)
act_b    = torch.randn(64, ACT_DIM).clamp(-1, 1)
rew_b    = torch.randn(64)
nobs_b   = torch.randn(64, OBS_DIM)
done_b   = torch.zeros(64)

c_losses = []
for _ in range(10):
    m = sac_test.update(obs_b, act_b, rew_b, nobs_b, done_b)
    c_losses.append(m["critic_loss"])

assert all(math.isfinite(v) for v in c_losses), "Métriques non finies"

# select_action
a = sac_test.select_action(np.zeros(OBS_DIM, dtype=np.float32))
assert a.shape == (ACT_DIM,), f"Shape action: {a.shape}"
assert np.all(np.abs(a) <= 1.1), "Action hors bornes"

print(f"✓ SAC mini-test OK | c_loss initial={c_losses[0]:.4f} | final={c_losses[-1]:.4f}")

## IV.3 — Les deux buffers et l'échantillonnage mixte

Rien d'exotique : deux files FIFO de transitions `(s, a, r, s', done)`, l'une pour le réel, l'autre pour
l'imaginé (capacité plus petite, on rafraîchit). Une fonction `sample_mixed` compose le batch
d'entraînement de SAC selon `real_ratio`.

In [ ]:
# ============================================================
# ReplayBuffer / ModelBuffer / sample_mixed
# ============================================================

class ReplayBuffer:
    def __init__(self, capacity):
        self._buf = collections.deque(maxlen=capacity)

    def push(self, obs, action, reward, next_obs, done):
        self._buf.append((
            np.asarray(obs, dtype=np.float32),
            np.asarray(action, dtype=np.float32),
            float(reward),
            np.asarray(next_obs, dtype=np.float32),
            float(done),
        ))

    def sample(self, batch_size):
        replace = batch_size > len(self._buf)
        idx = np.random.choice(len(self._buf), size=batch_size, replace=replace)
        s, a, r, ns, d = zip(*(self._buf[i] for i in idx))
        return (
            torch.tensor(np.array(s), dtype=torch.float32),
            torch.tensor(np.array(a), dtype=torch.float32),
            torch.tensor(np.array(r), dtype=torch.float32),
            torch.tensor(np.array(ns), dtype=torch.float32),
            torch.tensor(np.array(d), dtype=torch.float32),
        )

    def clear(self):
        self._buf.clear()

    def __len__(self):
        return len(self._buf)


class ModelBuffer(ReplayBuffer):
    # Même interface, capacité distincte (pour les transitions imaginées).
    pass


def sample_mixed(env_buf, model_buf, batch_size, real_ratio):
    # Compose un batch réel + imaginé selon real_ratio.
    env_empty = len(env_buf) == 0
    model_empty = len(model_buf) == 0
    if env_empty and model_empty:
        raise RuntimeError("Les deux buffers sont vides.")
    if env_empty:
        return model_buf.sample(batch_size)
    if model_empty:
        return env_buf.sample(batch_size)

    real_count = int(round(real_ratio * batch_size))
    real_count = max(1, min(real_count, batch_size - 1))
    model_count = batch_size - real_count

    real_batch = env_buf.sample(real_count)
    model_batch = model_buf.sample(model_count)

    return tuple(torch.cat([real_batch[i], model_batch[i]], dim=0) for i in range(5))


print("✓ ReplayBuffer / ModelBuffer / sample_mixed définis")

In [ ]:
# Mini-test : sample_mixed renvoie ~10 % de réel avec real_ratio=0.1

np.random.seed(0)

OBS_D_T, ACT_D_T = 4, 2
env_buf_t   = ReplayBuffer(capacity=5000)
model_buf_t = ModelBuffer(capacity=50000)

# Remplir avec des tags différents (obs[:,0] == 0 pour réel, 1 pour imaginé)
for _ in range(200):
    s = np.zeros(OBS_D_T, dtype=np.float32)
    env_buf_t.push(s, np.zeros(ACT_D_T), 0.0, s, False)

for _ in range(2000):
    s = np.ones(OBS_D_T, dtype=np.float32)
    model_buf_t.push(s, np.zeros(ACT_D_T), 0.0, s, False)

BATCH_T = 100
obs_m, *_ = sample_mixed(env_buf_t, model_buf_t, BATCH_T, real_ratio=0.1)
n_real_got = int((obs_m[:, 0] < 0.5).sum().item())  # obs[:,0]==0 → réel
assert 5 <= n_real_got <= 15, f"Attendu ~10 réels, obtenu {n_real_got}"

print(f"✓ sample_mixed OK | batch total={obs_m.shape[0]} | réels={n_real_got} (~10 %)")

## IV.4 — Le générateur de rollouts branchés

`generate_rollouts` est le cœur de MBPO — c'est lui qui transforme le modèle en **source de données**. Le principe « branché » (cf. II.3) : au lieu d'imaginer une longue trajectoire depuis l'état initial, on **tire un lot d'états réels** déjà visités (depuis $D_{\text{env}}$) et on n'imagine que **$k$ pas** à partir de chacun. Chaque rollout démarre donc d'un point **ancré dans la réalité**, jamais d'une accumulation d'hallucinations.

À chaque pas imaginé, pour tout le lot en parallèle :
1. la **politique courante** propose une action $a$ depuis l'état $s$ ;
2. on **tire au hasard un membre** de l'ensemble — un par ligne — qui prédit la gaussienne sur $[\Delta s,\ r]$, dont on échantillonne $s' = s + \Delta s$ et la récompense $r$ ;
3. la transition $(s, a, r, s')$ atterrit dans $D_{\text{model}}$ ;
4. on repart de $s'$ pour le pas suivant.

Deux détails qui comptent. Le **membre tiré au hasard** (*trajectory sampling*) fait diverger les rollouts là où les membres de l'ensemble sont en **désaccord** : l'incertitude épistémique se propage ainsi gratuitement dans les données imaginées. Et `done` reste **toujours `False`** : sur un rollout aussi court on ne modélise pas de terminaison, on accumule juste de **courts fragments de dynamique** réutilisables par SAC.


In [ ]:
@torch.no_grad()
def generate_rollouts(env_buf, model_buf, ensemble, actor, k, n_states):
    # Génère k pas de rollouts branchés depuis n_states états réels
    # Retourne un dict de diagnostics
    obs_sample, *_ = env_buf.sample(min(n_states, len(env_buf)))
    states = obs_sample.clone()  # [n_states, obs_dim]
    B = states.shape[0]
    total_rew = 0.0

    for _ in range(k):
        # Actions batch via la politique courante
        acts_raw, _, _ = actor.actor.sample(states)   # [B, act_dim]
        acts = acts_raw                                 # tanh ∈ [-1,1], adapté à HalfCheetah

        # Tirer un membre de l'ensemble aléatoirement pour chaque ligne
        model_idx = torch.randint(0, ensemble.ensemble_size, (B,))

        # Propager dans le modèle
        next_states, rewards = ensemble.propagate(states, acts, model_idx)

        # Pousser les transitions dans model_buf
        for i in range(B):
            model_buf.push(
                states[i].numpy(),
                acts[i].numpy(),
                float(rewards[i].item()),
                next_states[i].numpy(),
                False,
            )
        total_rew += float(rewards.mean().item())
        states = next_states

    return {
        "model_buf_size":  len(model_buf),
        "mean_imagined_r": total_rew / k if k > 0 else 0.0,
    }


print("✓ generate_rollouts défini")

## IV.5 — Assembler le modèle, SAC et les deux buffers


In [ ]:
class MBPOAgent:
    def __init__(
        self, obs_dim, action_dim, action_low, action_high,
        *, model_hidden, sac_hidden, ensemble_size, env_buffer_capacity,
        model_buffer_capacity,
    ):
        self.ensemble = ProbabilisticEnsemble(
            input_dim=obs_dim + action_dim,
            output_dim=obs_dim + 1,
            ensemble_size=ensemble_size,
            hidden_dim=model_hidden,
            n_layers=3,
        )
        self.sac = SacLearner(obs_dim, action_dim, hidden_dim=sac_hidden)
        self.env_buffer = ReplayBuffer(capacity=env_buffer_capacity)
        self.model_buffer = ModelBuffer(capacity=model_buffer_capacity)
        self.action_scale = (action_high - action_low) / 2.0
        self.action_bias = (action_high + action_low) / 2.0

    def to_env_action(self, normalized_action):
        clipped = np.clip(normalized_action, -1.0, 1.0)
        return (clipped * self.action_scale + self.action_bias).astype(np.float32)

    def select_action(self, observation, deterministic=False):
        return self.sac.select_action(observation, deterministic=deterministic)

    def store_real_transition(self, observation, action, reward, next_observation, done):
        self.env_buffer.push(observation, action, reward, next_observation, done)

    def _model_dataset(self):
        rows = self.env_buffer._buf
        inputs = torch.tensor(
            np.array([[*row[0], *row[1]] for row in rows]), dtype=torch.float32,
        )
        targets = torch.tensor(
            np.array([[*(row[3] - row[0]), row[2]] for row in rows]), dtype=torch.float32,
        )
        return inputs, targets

    def fit_model(self, *, steps, batch_size=256):
        inputs, targets = self._model_dataset()
        return self.ensemble.fit(inputs, targets, steps=steps, batch_size=batch_size)

    def generate_model_rollouts(self, rollout_length, n_states):
        return generate_rollouts(
            self.env_buffer, self.model_buffer, self.ensemble, self.sac,
            k=rollout_length, n_states=n_states,
        )

    def update_policy(self, batch_size, real_ratio, updates):
        # Avant les premiers rollouts imaginés, le notebook original fait une seule update réelle.
        n_updates = updates if len(self.model_buffer) >= batch_size else 1
        for _ in range(n_updates):
            if len(self.model_buffer) >= batch_size:
                batch = sample_mixed(
                    self.env_buffer, self.model_buffer, batch_size, real_ratio,
                )
            else:
                batch = self.env_buffer.sample(batch_size)
            self.sac.update(*batch)


print("MBPOAgent défini : modèle, SAC et buffers assemblés")

In [ ]:
# Mini-test + visualisation : ce que generate_rollouts fabrique vraiment
# (état jouet en 2D pour un plot lisible ; dynamique = spirale amortie apprenable)
torch.manual_seed(3)
np.random.seed(3)

OBS_DT, ACT_DT = 2, 1

# Dynamique jouet : spirale amortie  s' = 0.92 * R(theta) @ s + petit effet de l'action
THETA = 0.30
R = np.array([[np.cos(THETA), -np.sin(THETA)],
              [np.sin(THETA),  np.cos(THETA)]], dtype=np.float32)
def toy_step(s, a):
    return 0.92 * (R @ s) + np.array([0.10 * a[0], 0.0], dtype=np.float32)

ENS   = ProbabilisticEnsemble(OBS_DT + ACT_DT, OBS_DT + 1, ensemble_size=5, hidden_dim=64, n_layers=2)
ACT_T = SacLearner(OBS_DT, ACT_DT, hidden_dim=32)
ENV_BT   = ReplayBuffer(capacity=2000)
MODEL_BT = ModelBuffer(capacity=5000)

# Remplir env_buf avec de vraies transitions de la dynamique jouet
for _ in range(400):
    s  = np.random.uniform(-2.0, 2.0, OBS_DT).astype(np.float32)
    a  = np.random.uniform(-1.0, 1.0, ACT_DT).astype(np.float32)
    ns = toy_step(s, a) + 0.02 * np.random.randn(OBS_DT).astype(np.float32)
    r  = -float(s @ s)                          # un coût-like quelconque (tête de récompense)
    ENV_BT.push(s, a, r, ns, False)

# Entraîner l'ensemble sur [Δs, r]
X_t = torch.tensor(np.array([[*ENV_BT._buf[i][0], *ENV_BT._buf[i][1]]
                              for i in range(len(ENV_BT._buf))]), dtype=torch.float32)
Y_t = torch.tensor(np.array([[*(ENV_BT._buf[i][3] - ENV_BT._buf[i][0]), ENV_BT._buf[i][2]]
                              for i in range(len(ENV_BT._buf))]), dtype=torch.float32)
ENS.fit(X_t, Y_t, steps=150, batch_size=64)

# ── Test structurel (inchangé dans l'esprit) ──────────────────────────────
len_before = len(MODEL_BT)
generate_rollouts(ENV_BT, MODEL_BT, ENS, ACT_T, k=1, n_states=16)
assert len(MODEL_BT) == len_before + 16, f"Attendu +16, obtenu +{len(MODEL_BT) - len_before}"
s_t, a_t, r_t, ns_t, d_t = MODEL_BT.sample(8)
assert s_t.shape == (8, OBS_DT) and a_t.shape == (8, ACT_DT)
assert r_t.shape == (8,)        and ns_t.shape == (8, OBS_DT)

# ── Visualisation ─────────────────────────────────────────────────────────
@torch.no_grad()
def imagine_branch(start_states, k):
    # reproduit la logique de generate_rollouts, mais garde la trajectoire pour le plot
    states = torch.as_tensor(np.array(start_states), dtype=torch.float32)
    traj = [states.numpy().copy()]
    for _ in range(k):
        acts, _, _ = ACT_T.actor.sample(states)
        midx = torch.randint(0, ENS.ensemble_size, (states.shape[0],))
        states, _ = ENS.propagate(states, acts, midx)
        traj.append(states.numpy().copy())
    return np.stack(traj, axis=0)   # [k+1, B, obs_dim]

real_s = np.array([ENV_BT._buf[i][0] for i in range(len(ENV_BT._buf))])
starts, *_ = ENV_BT.sample(12)
branch = imagine_branch(starts.numpy(), k=8)   # [9, 12, 2]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))

# Panneau 1 : rollouts branchés
ax = axes[0]
ax.scatter(real_s[:, 0], real_s[:, 1], s=8, c="lightgray", label=r"états réels ($D_{env}$)")
for b in range(branch.shape[1]):
    ax.plot(branch[:, b, 0], branch[:, b, 1], "-", lw=1.3, alpha=0.85, color="steelblue")
ax.scatter(branch[0, :, 0],  branch[0, :, 1],  s=60, c="crimson", edgecolor="black",
           zorder=5, label="départ réel")
ax.scatter(branch[-1, :, 0], branch[-1, :, 1], s=28, c="steelblue", zorder=5,
           label="fin imaginée ($k=8$)")
ax.set_title("Rollouts branchés : $k$ pas imaginés depuis de vrais états")
ax.set_xlabel("$s_0$"); ax.set_ylabel("$s_1$"); ax.legend(fontsize=8)

# Panneau 2 : désaccord de l'ensemble depuis un même état
ax = axes[1]
s0 = torch.as_tensor(real_s[np.argmax(np.abs(real_s).sum(1))], dtype=torch.float32).unsqueeze(0)
a0, _, _ = ACT_T.actor.sample(s0)
means, _ = ENS.predict(torch.cat([s0, a0], dim=-1))           # [1, E, obs+1]
member_ns = s0.numpy() + means[0, :, :OBS_DT].numpy()         # moyenne de chaque membre
cloud = []
for _ in range(300):                                          # comme generate_rollouts : membre random + bruit
    midx = torch.randint(0, ENS.ensemble_size, (1,))
    ns, _ = ENS.propagate(s0, a0, midx)
    cloud.append(ns.numpy()[0])
cloud = np.array(cloud)
ax.scatter(cloud[:, 0], cloud[:, 1], s=10, c="lightsteelblue", alpha=0.5, label=r"$s'$ échantillonnés")
ax.scatter(member_ns[:, 0], member_ns[:, 1], s=70, c="crimson", marker="D", edgecolor="black",
           zorder=5, label="moyenne par membre")
ax.scatter(s0.numpy()[0, 0], s0.numpy()[0, 1], s=120, c="black", marker="*",
           zorder=6, label="état de départ $s$")
ax.set_title(r"Désaccord de l'ensemble : un état $\to$ un nuage de $s'$")
ax.set_xlabel("$s_0$"); ax.set_ylabel("$s_1$"); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

print(f"✓ generate_rollouts OK | +16 transitions, shapes OK | 12 branches (k=8) tracées")

**Lecture.** À gauche, chaque rollout imaginé (bleu) **part d'un vrai état** (rouge) pris dans $D_{\text{env}}$ et ne s'éloigne que de quelques pas : les branches restent **collées au nuage réel** plutôt que de filer vers des régions jamais visitées — c'est tout l'intérêt des rollouts *branchés et courts*. À droite, depuis un **même** état, les membres de l'ensemble ne prédisent pas la même chose (losanges) et l'échantillonnage produit un **nuage** de $s'$ : c'est l'**incertitude** du modèle rendue visible. `generate_rollouts` tire un membre au hasard à chaque pas, donc les trajectoires explorent naturellement ce désaccord — là où l'ensemble hésite, les rêves se dispersent, et la politique apprend à ne pas trop s'y fier.

In [ ]:
@torch.no_grad()
def evaluate_greedy(env_name, sac, episodes=3, max_steps=1000, seed=1234):
    """Éval déterministe (greedy) de la politique : action = tanh(mean), sans bruit.

    Sert aux courbes d'apprentissage propres pendant le training.
    La démonstration visuelle est séparée et utilise `RecordVideo`.
    """
    env = gym.make(env_name, max_episode_steps=max_steps)
    low  = env.action_space.low.astype(np.float32)
    high = env.action_space.high.astype(np.float32)
    scale = (high - low) / 2.0
    bias  = (high + low) / 2.0
    returns = []
    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + ep)
        ep_ret = 0.0
        for _ in range(max_steps):
            a_norm = sac.select_action(obs.astype(np.float32), deterministic=True)
            a_env = (np.clip(a_norm, -1.0, 1.0) * scale + bias).astype(np.float32)
            obs, r, terminated, truncated, _ = env.step(a_env)
            ep_ret += float(r)
            if terminated or truncated:
                break
        returns.append(ep_ret)
    env.close()
    return float(np.mean(returns)), float(np.std(returns))


print("✓ evaluate_greedy défini")


## IV.5 — La boucle MBPO complète

$$
\boxed{
\begin{aligned}
&\textbf{Entrée : } \text{env réel},\ \rho=\texttt{real\_ratio},\ G\ \text{updates/pas},\ k_{\max} \\
&\textbf{Init : } \text{ensemble } p_\theta,\ \text{politique SAC } \pi_\phi,\ \text{buffers } \mathcal{D}_{\text{env}},\ \mathcal{D}_{\text{model}} \\
&\textbf{Warmup : } \text{remplir } \mathcal{D}_{\text{env}} \text{ avec une politique aléatoire} \\
&\textbf{pour chaque} \text{ époque :} \\
&\quad \text{(1) entraîner } p_\theta \text{ sur } \mathcal{D}_{\text{env}} \quad (\text{NLL sur } [\Delta s,\, r]) \\
&\quad \text{(2) } k \leftarrow \text{schedule(époque)} \\
&\quad \textbf{pour chaque} \text{ pas réel :} \\
&\quad\quad \text{(a) } a \sim \pi_\phi(\cdot \mid s);\quad s', r \leftarrow \text{env.step}(a);\quad (s,a,r,s',\text{done}) \to \mathcal{D}_{\text{env}} \\
&\quad\quad \text{(b) périodiquement : } k \text{ pas branchés depuis états réels} \to \mathcal{D}_{\text{model}} \\
&\quad\quad \text{(c) répéter } G \text{ fois : update SAC sur } \texttt{sample\_mixed}(\mathcal{D}_{\text{env}}, \mathcal{D}_{\text{model}}, \rho) \\
\end{aligned}
}
$$

Une seule ligne touche le vrai monde — l'étape **(a)**. Tout le reste — fit du modèle, rollouts imaginés, $G$ updates SAC — se passe « hors-ligne ». C'est là toute la mécanique de l'efficacité-échantillon.

> **Le `done` qu'on stocke — `terminated`, pas `truncated`.** Le pseudo-code stocke `done = float(terminated)`, **jamais** `float(terminated or truncated)`. Une fin par **limite de temps** (`truncated`) n'est pas un état terminal du MDP : l'épisode est coupé arbitrairement, donc le bootstrap doit rester actif, $y = r + \gamma\,(1-\text{done})\,\min_i Q_{\phi'_i}$. Comme HalfCheetah-v5 et Pendulum-v1 ne terminent **jamais** (ils ne font que tronquer à `max_ep_steps`), stocker `done=1` en fin d'épisode sous-estimerait *systématiquement* les valeurs de fin d'épisode. On remet donc l'environnement à zéro sur `terminated or truncated`, mais on ne coupe le bootstrap que sur `terminated`. C'est exactement la convention détaillée dans le [notebook 08 — SAC](08_sac_halfcheetah_walkthrough.ipynb) (note « `terminated` vs `truncated` »).

> **Sur `k` (longueur de rollout).** Le pseudo-code écrit `k ← schedule(époque)` : `run_mbpo` câble bien le `k_schedule` de II.4 (via `rollout_length_max`), mais on le laisse **constant à `k=1`** dans nos démos courtes — le modèle n'a pas le temps de devenir assez fiable pour mériter des rollouts plus longs. Fixer `rollout_length_max > 1` réactive la croissance.

In [ ]:
def run_mbpo(
    env_name="HalfCheetah-v5",
    total_real_steps=2000,
    warmup_steps=500,
    rollout_length=1,
    rollout_length_max=None,   # si > rollout_length : k croît selon le schedule de II.4 ; None => k constant
    rollout_grow_epochs=(0, 10),
    rollout_every=50,
    rollout_batch=300,
    updates_per_step=20,
    real_ratio=0.05,
    batch_size=256,
    model_hidden=128,
    sac_hidden=128,
    ensemble_size=5,
    model_fit_steps=150,
    env_buf_cap=100_000,
    model_buf_cap=100_000,
    max_ep_steps=200,
    eval_every=0,
    eval_episodes=3,
    seed=0,
):
    np.random.seed(seed)
    torch.manual_seed(seed)

    env = gym.make(env_name, max_episode_steps=max_ep_steps)
    env.action_space.seed(seed)

    try:
        obs_dim = env.observation_space.shape[0]
        act_dim = env.action_space.shape[0]
        agent = MBPOAgent(
            obs_dim, act_dim,
            env.action_space.low.astype(np.float32),
            env.action_space.high.astype(np.float32),
            model_hidden=model_hidden,
            sac_hidden=sac_hidden,
            ensemble_size=ensemble_size,
            env_buffer_capacity=env_buf_cap,
            model_buffer_capacity=model_buf_cap,
        )

        model_refit_period = rollout_every * 5

        ep_returns = []       # (real_step, ep_return)
        nll_history = []      # (real_step, nll)
        eval_history = []     # (real_step, greedy_return)
        rollout_history = []  # (real_step, k, model_buffer_size)

        print(
            f"MBPO | env={env_name} | warmup={warmup_steps} | real_steps={total_real_steps} | "
            f"ensemble={ensemble_size} | rollout_every={rollout_every} | updates/step={updates_per_step}"
        )

        # ── Warmup : politique aléatoire ───────────────────────────────
        obs, _ = env.reset(seed=seed)
        ep_ret = 0.0

        warmup_bar = tqdm(range(warmup_steps), desc="Warmup aléatoire", leave=False)
        for _ in warmup_bar:
            a_norm = np.random.uniform(-1.0, 1.0, size=act_dim).astype(np.float32)
            next_obs, r, terminated, truncated, _ = env.step(agent.to_env_action(a_norm))

            # truncated = limite de temps, pas terminal dynamique : on garde le bootstrap côté SAC.
            done = float(terminated)
            agent.store_real_transition(obs, a_norm, float(r), next_obs, done)

            ep_ret += float(r)
            obs = next_obs

            if terminated or truncated:
                ep_returns.append((len(agent.env_buffer), ep_ret))
                warmup_bar.set_postfix(
                    ep_ret=f"{ep_ret:+.1f}",
                    env_buf=len(agent.env_buffer),
                )
                obs, _ = env.reset()
                ep_ret = 0.0

        print(f"Warmup terminé : {len(agent.env_buffer)} transitions réelles collectées")

        # ── Premier fit du modèle ───────────────────────────────────────
        real_step = len(agent.env_buffer)
        print(f"Fit initial du modèle probabiliste ({model_fit_steps} updates)...")
        nll = agent.fit_model(steps=model_fit_steps, batch_size=256)
        nll_history.append((real_step, nll))
        print(f"  NLL initiale : {nll:.3f}")

        # ── Boucle principale ──────────────────────────────────────────
        obs, _ = env.reset()
        ep_ret = 0.0
        last_rollout_step = real_step
        target_step = warmup_steps + total_real_steps

        last_eval = None
        last_k = rollout_length
        last_ep_ret = ep_returns[-1][1] if ep_returns else None

        pbar = tqdm(
            total=total_real_steps,
            initial=max(0, real_step - warmup_steps),
            desc="MBPO training",
        )

        while real_step < target_step:
            # a) Interagir dans le vrai monde.
            a_norm = agent.select_action(obs.astype(np.float32))  # politique normalisée en [-1, 1]
            next_obs, r, terminated, truncated, _ = env.step(agent.to_env_action(a_norm))

            done = float(terminated)
            agent.store_real_transition(obs, a_norm, float(r), next_obs, done)

            ep_ret += float(r)
            obs = next_obs
            real_step += 1
            pbar.update(1)

            if terminated or truncated:
                ep_returns.append((real_step, ep_ret))
                last_ep_ret = ep_ret
                obs, _ = env.reset()
                ep_ret = 0.0

            # b) Évaluation greedy périodique : courbe propre, sans bruit d'exploration.
            if eval_every and real_step % eval_every == 0:
                gm, gs = evaluate_greedy(
                    env_name,
                    agent.sac,
                    episodes=eval_episodes,
                    max_steps=max_ep_steps,
                    seed=10_000,
                )
                eval_history.append((real_step, gm))
                last_eval = gm
                tqdm.write(f"[eval @ {real_step}] greedy={gm:+.1f} ± {gs:.1f}")

            # c) Rollouts branchés depuis des états réels.
            if real_step - last_rollout_step >= rollout_every:
                if rollout_length_max is None:
                    k_now = rollout_length
                else:
                    epoch = (real_step - warmup_steps) // model_refit_period
                    k_now = k_schedule(
                        epoch,
                        rollout_length,
                        rollout_length_max,
                        rollout_grow_epochs[0],
                        rollout_grow_epochs[1],
                    )

                agent.generate_model_rollouts(k_now, rollout_batch)
                rollout_history.append((real_step, k_now, len(agent.model_buffer)))
                last_rollout_step = real_step
                last_k = k_now

            # d) Updates SAC : réel seul au début, puis batch mixte réel/modèle.
            if len(agent.env_buffer) >= batch_size:
                agent.update_policy(batch_size, real_ratio, updates_per_step)

            # e) Refit périodique du modèle de dynamique.
            if real_step % model_refit_period == 0:
                tqdm.write(f"[model fit @ {real_step}] {model_fit_steps} updates...")
                nll = agent.fit_model(steps=model_fit_steps, batch_size=256)
                nll_history.append((real_step, nll))
                tqdm.write(f"[model fit @ {real_step}] NLL={nll:.3f}")

            pbar.set_postfix(
                ep_ret="-" if last_ep_ret is None else f"{last_ep_ret:+.1f}",
                nll=f"{nll_history[-1][1]:+.2f}",
                eval="-" if last_eval is None else f"{last_eval:+.1f}",
                k=last_k,
                real_buf=len(agent.env_buffer),
                model_buf=len(agent.model_buffer),
            )

        pbar.close()

        print(
            f"MBPO terminé : real_buf={len(agent.env_buffer)}, "
            f"model_buf={len(agent.model_buffer)}, épisodes={len(ep_returns)}, "
            f"refits={len(nll_history)}, evals={len(eval_history)}"
        )

        return {
            "ep_returns": ep_returns,
            "nll_history": nll_history,
            "eval_history": eval_history,
            "rollout_history": rollout_history,
            "agent": agent,
            "sac": agent.sac,
            "label": "MBPO",
        }

    finally:
        env.close()


print("✓ run_mbpo défini avec suivi tqdm")

On a tout. Reste à **prouver** la promesse de MBPO : apprendre plus vite *par pas réel* qu'un SAC seul.
Pour cela, un protocole équitable — même budget de vraies interactions pour les deux.

---
# Partie V — La preuve : MBPO vs SAC à budget de pas réels égal

## V.1 — Un protocole équitable

L'efficacité-échantillon se mesure **par interaction réelle**. On lance donc deux apprentissages sur le
**même petit budget de pas réels** :

- **SAC seul** (model-free) : une mise à jour par pas réel, sur données réelles uniquement.
- **MBPO** : le même SAC, mais nourri en plus de rollouts imaginés et entraîné $G$ fois par pas réel.

On trace le **retour réel** en fonction du **nombre de pas réels** consommés. Si MBPO grimpe plus vite,
c'est qu'il extrait davantage de chaque vraie interaction. *(Réglages volontairement minuscules : on
montre la mécanique et la tendance, pas un score de référence.)*

Point crucial qu'on va **découvrir empiriquement** : l'avantage de MBPO **n'est pas automatique** — il
dépend de la *qualité du modèle*. On teste donc deux régimes : d'abord **HalfCheetah** (17 dimensions,
difficile à modéliser), puis **Pendulum** (3 dimensions, facile). Le contraste *est* la leçon.

In [ ]:
def run_sac_baseline(
    env_name="HalfCheetah-v5",
    total_real_steps=2000,
    warmup_steps=500,
    sac_hidden=128,
    batch_size=256,
    env_buf_cap=100_000,
    max_ep_steps=200,
    eval_every=0,
    eval_episodes=3,
    seed=0,
):
    np.random.seed(seed)
    torch.manual_seed(seed)

    env = gym.make(env_name, max_episode_steps=max_ep_steps)
    env.action_space.seed(seed)

    try:
        obs_dim = env.observation_space.shape[0]
        act_dim = env.action_space.shape[0]
        act_low = env.action_space.low.astype(np.float32)
        act_high = env.action_space.high.astype(np.float32)
        act_scale = (act_high - act_low) / 2.0
        act_bias = (act_high + act_low) / 2.0

        def to_env(a_norm):
            # La politique sort dans [-1,1] (tanh) ; on remet à l'échelle de l'env [low, high].
            return (np.clip(a_norm, -1.0, 1.0) * act_scale + act_bias).astype(np.float32)

        sac = SacLearner(obs_dim, act_dim, hidden_dim=sac_hidden)
        env_buf = ReplayBuffer(capacity=env_buf_cap)

        ep_returns = []
        eval_history = []  # (real_step, greedy_return)

        print(
            f"SAC baseline | env={env_name} | warmup={warmup_steps} | "
            f"real_steps={total_real_steps} | hidden={sac_hidden}"
        )

        # ── Warmup aléatoire ──────────────────────────────────────────
        obs, _ = env.reset(seed=seed)
        ep_ret = 0.0

        warmup_bar = tqdm(range(warmup_steps), desc="Warmup SAC aléatoire", leave=False)
        for _ in warmup_bar:
            a_norm = np.random.uniform(-1.0, 1.0, size=act_dim).astype(np.float32)
            next_obs, r, terminated, truncated, _ = env.step(to_env(a_norm))

            # truncated = limite de temps, pas terminal dynamique : on garde le bootstrap côté SAC.
            done = float(terminated)
            env_buf.push(obs, a_norm, float(r), next_obs, done)

            ep_ret += float(r)
            obs = next_obs

            if terminated or truncated:
                ep_returns.append((len(env_buf), ep_ret))
                warmup_bar.set_postfix(
                    ep_ret=f"{ep_ret:+.1f}",
                    env_buf=len(env_buf),
                )
                obs, _ = env.reset()
                ep_ret = 0.0

        print(f"Warmup terminé : {len(env_buf)} transitions réelles collectées")

        # ── Boucle principale ──────────────────────────────────────────
        real_step = len(env_buf)
        target_step = warmup_steps + total_real_steps
        obs, _ = env.reset()
        ep_ret = 0.0

        last_ep_ret = ep_returns[-1][1] if ep_returns else None
        last_eval = None
        update_count = 0

        pbar = tqdm(
            total=total_real_steps,
            initial=max(0, real_step - warmup_steps),
            desc="SAC baseline training",
        )

        while real_step < target_step:
            a_norm = sac.select_action(obs.astype(np.float32))  # politique en [-1,1]
            next_obs, r, terminated, truncated, _ = env.step(to_env(a_norm))

            done = float(terminated)
            env_buf.push(obs, a_norm, float(r), next_obs, done)

            ep_ret += float(r)
            obs = next_obs
            real_step += 1
            pbar.update(1)

            if terminated or truncated:
                ep_returns.append((real_step, ep_ret))
                last_ep_ret = ep_ret
                obs, _ = env.reset()
                ep_ret = 0.0

            # Éval greedy périodique : courbe d'apprentissage propre, déterministe.
            if eval_every and real_step % eval_every == 0:
                gm, gs = evaluate_greedy(
                    env_name,
                    sac,
                    episodes=eval_episodes,
                    max_steps=max_ep_steps,
                    seed=10_000,
                )
                eval_history.append((real_step, gm))
                last_eval = gm
                tqdm.write(f"[SAC eval @ {real_step}] greedy={gm:+.1f} ± {gs:.1f}")

            # 1 update SAC par pas réel après warmup.
            if len(env_buf) >= batch_size:
                o, a_s, rw, no, do = env_buf.sample(batch_size)
                sac.update(o, a_s, rw, no, do)
                update_count += 1

            pbar.set_postfix(
                ep_ret="-" if last_ep_ret is None else f"{last_ep_ret:+.1f}",
                eval="-" if last_eval is None else f"{last_eval:+.1f}",
                env_buf=len(env_buf),
                updates=update_count,
            )

        pbar.close()

        print(
            f"SAC baseline terminé : real_buf={len(env_buf)}, "
            f"épisodes={len(ep_returns)}, updates={update_count}, evals={len(eval_history)}"
        )

        return {
            "ep_returns": ep_returns,
            "eval_history": eval_history,
            "sac": sac,
            "label": "SAC baseline",
        }

    finally:
        env.close()


print("✓ run_sac_baseline défini avec suivi tqdm")

## V.2 — Premier essai : HalfCheetah (le test honnête)

Commençons par notre environnement de référence, HalfCheetah. On donne à MBPO un **UTD modéré** et un
petit budget de pas réels, et on le compare à SAC seul sur le même budget.

In [ ]:
# Protocole HalfCheetah : MBPO et SAC reçoivent le même budget réel.
# On garde un warmup plus large pour que le modèle ait un support initial moins pauvre.
HC_REAL_STEPS = 20_000
WARMUP = 5_000
MAX_EP_STEPS = 1000
SEED = 0
HC_EVAL_EVERY = 2_000

np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# HalfCheetah : tâche standard (max_ep_steps=1000).
# Le signal propre vient de l'éval greedy périodique, pas des retours stochastiques bruités.

def final_return(results, n=5):
    rets = [r for _, r in results["ep_returns"]]
    if not rets:
        return 0.0
    return float(np.mean(rets[-n:])) if len(rets) >= n else float(np.mean(rets))


print("Lancement MBPO (HalfCheetah)...")
t0 = time.time()
mbpo_results = run_mbpo(
    env_name="HalfCheetah-v5",
    total_real_steps=HC_REAL_STEPS,
    warmup_steps=WARMUP,

    # Rollouts courts et fréquents : on veut beaucoup d'imaginé frais, mais peu profond.
    rollout_length=1,
    rollout_every=100,
    rollout_batch=800,

    # Moins agressif que 10 updates/pas + 5 % réel : on évite de graver trop vite les biais du modèle.
    updates_per_step=5,
    real_ratio=0.20,
    batch_size=256,

    # Modèle un peu plus capable, mais encore raisonnable pour notebook.
    model_hidden=200,
    sac_hidden=256,
    ensemble_size=5,
    model_fit_steps=200,

    # D_env conserve le réel ; D_model est un FIFO court de rollouts synthétiques récents.
    env_buf_cap=100_000,
    model_buf_cap=10_000,

    max_ep_steps=MAX_EP_STEPS,
    seed=SEED,
    eval_every=HC_EVAL_EVERY,
    eval_episodes=3,
)
print(f"  MBPO terminé en {time.time()-t0:.1f}s")

print("Lancement SAC baseline (HalfCheetah)...")
t1 = time.time()
sac_results = run_sac_baseline(
    env_name="HalfCheetah-v5",
    total_real_steps=HC_REAL_STEPS,
    warmup_steps=WARMUP,
    sac_hidden=256,
    batch_size=256,
    max_ep_steps=MAX_EP_STEPS,
    seed=SEED,
    eval_every=HC_EVAL_EVERY,
    eval_episodes=3,
)
print(f"  SAC terminé en  {time.time()-t1:.1f}s")

print(f"\nHalfCheetah — retour greedy final  MBPO : {mbpo_results['eval_history'][-1][1]:.1f}")
print(f"HalfCheetah — retour greedy final  SAC  : {sac_results['eval_history'][-1][1]:.1f}")

In [ ]:
# Courbe d'apprentissage : retour GREEDY (déterministe) vs pas réels — MBPO vs SAC
# (les retours stochastiques bruts restent en fond, en points pâles)
def extract_curve(results):
    steps_arr = np.array([s for s, _ in results["ep_returns"]], dtype=float)
    rets_arr  = np.array([r for _, r in results["ep_returns"]], dtype=float)
    return steps_arr, rets_arr

def extract_eval(results):
    ev = np.array(results["eval_history"], dtype=float)
    return (ev[:, 0], ev[:, 1]) if len(ev) else (np.array([]), np.array([]))

steps_m, rets_m = extract_curve(mbpo_results)
steps_s, rets_s = extract_curve(sac_results)
ev_m_x, ev_m_y = extract_eval(mbpo_results)
ev_s_x, ev_s_y = extract_eval(sac_results)

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(steps_m, rets_m, alpha=0.20, s=14, color="steelblue")
ax.scatter(steps_s, rets_s, alpha=0.20, s=14, color="coral")
ax.plot(ev_m_x, ev_m_y, color="steelblue", lw=2.5, marker="o", label="MBPO (greedy)")
ax.plot(ev_s_x, ev_s_y, color="coral",     lw=2.5, marker="o", label="SAC (greedy)")
ax.set_xlabel("Pas réels")
ax.set_ylabel("Retour par épisode")
ax.set_title("HalfCheetah : MBPO vs SAC (retour greedy, budget de pas réels égal)")
ax.legend()
plt.tight_layout()
plt.show()


**Lecture.** Cette fois, l'avantage MBPO est net : à budget de pas réels égal, la politique entraînée
avec rollouts imaginés atteint un retour greedy final autour de **1940**, contre environ **650** pour
SAC seul. Le guépard ne se contente donc plus de bouger : il apprend une vraie locomotion vers l'avant,
et l'imagination accélère clairement l'apprentissage.

Ce résultat dépend beaucoup des garde-fous introduits dans la config corrigée. On garde des rollouts
**très courts** (`k=1`), un `D_model` en **FIFO court** pour oublier les imaginations trop anciennes,
un `real_ratio` plus élevé pour ancrer SAC dans le réel, et un nombre d'updates par pas réel moins
agressif. Autrement dit, MBPO gagne parce qu'on utilise le modèle là où il est fiable : près des états
réels, sur un horizon court, avec un replay imaginé récent.

La leçon est exactement celle du papier *When to Trust Your Model* : un modèle imparfait peut être très
utile si on limite son domaine d'utilisation. Ici, l'imagination ne remplace pas le réel ; elle le
**multiplie localement**. Chaque vraie transition sert à ajuster le modèle, puis ce modèle produit de
courtes transitions supplémentaires qui donnent à SAC beaucoup plus de signal d'entraînement que le
SAC baseline.

In [ ]:
# Diagnostic : est-ce que le replay imaginé a une échelle de reward plausible ?

def summarize_buffer_rewards(buffer, n=3000, seed=0):
    if len(buffer) == 0:
        return None

    rng = np.random.default_rng(seed)
    m = min(n, len(buffer))
    idx = rng.choice(len(buffer._buf), size=m, replace=False)

    rewards = np.array([buffer._buf[i][2] for i in idx], dtype=np.float32)
    dones = np.array([buffer._buf[i][4] for i in idx], dtype=np.float32)

    return {
        "mean": float(rewards.mean()),
        "std": float(rewards.std()),
        "p05": float(np.percentile(rewards, 5)),
        "p50": float(np.percentile(rewards, 50)),
        "p95": float(np.percentile(rewards, 95)),
        "done_mean": float(dones.mean()),
    }


env_r_stats = summarize_buffer_rewards(mbpo_results["agent"].env_buffer)
model_r_stats = summarize_buffer_rewards(mbpo_results["agent"].model_buffer)

print("Reward dans les buffers MBPO :")
for name, stats in [("réel", env_r_stats), ("imaginé récent", model_r_stats)]:
    if stats is None:
        print(f"  {name:13s} | buffer vide")
        continue
    print(
        f"  {name:13s} | mean={stats['mean']:+.3f} | std={stats['std']:.3f} "
        f"| p05={stats['p05']:+.3f} | p50={stats['p50']:+.3f} | p95={stats['p95']:+.3f} "
        f"| done={stats['done_mean']:.3f}"
    )

real_rewards = np.array([row[2] for row in mbpo_results["agent"].env_buffer._buf], dtype=np.float32)
model_rewards = np.array([row[2] for row in mbpo_results["agent"].model_buffer._buf], dtype=np.float32)

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.hist(real_rewards, bins=60, alpha=0.45, density=True, label="réel", color="steelblue")
ax.hist(model_rewards, bins=60, alpha=0.45, density=True, label="imaginé récent", color="coral")
ax.axvline(real_rewards.mean(), color="steelblue", lw=2, ls="--")
ax.axvline(model_rewards.mean(), color="coral", lw=2, ls="--")
ax.set_title("Distribution des rewards : réel vs imaginé récent")
ax.set_xlabel("reward one-step")
ax.set_ylabel("densité")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Lecture.** Le replay imaginé doit rester du même ordre de grandeur que le replay réel. S'il est
beaucoup plus optimiste, SAC apprend à exploiter le modèle plutôt que HalfCheetah. S'il est beaucoup
plus pessimiste ou très dispersé, il agit comme un bruit d'entraînement qui peut ralentir SAC. Le point
important est que `D_model` est **off-policy mais périssable** : on ne le vide pas à chaque refit, mais
on le garde en FIFO court (`model_buf_cap=10_000`) pour conserver quelques vagues récentes de rollouts
et oublier progressivement les imaginations trop anciennes.

In [ ]:
# Diagnostic : combien d'imaginé SAC voit-il réellement ?

def mbpo_data_usage_summary(results, batch_size=256, real_ratio=0.20, updates_per_step=5):
    agent = results["agent"]
    real_per_update = int(round(real_ratio * batch_size))
    model_per_update = batch_size - real_per_update

    print("Utilisation typique des données par update SAC MBPO :")
    print(f"  batch_size        : {batch_size}")
    print(f"  real_ratio        : {real_ratio:.2f}")
    print(f"  réel/update       : {real_per_update}")
    print(f"  imaginé/update    : {model_per_update}")
    print(f"  updates/pas réel  : {updates_per_step}")
    print(f"  env_buffer        : {len(agent.env_buffer)} transitions")
    print(f"  model_buffer FIFO : {len(agent.model_buffer)} / {agent.model_buffer._buf.maxlen} transitions")

    imagined_fraction = model_per_update / batch_size
    print(f"\nDonc environ {100 * imagined_fraction:.1f}% de chaque batch SAC vient encore du modèle.")


mbpo_data_usage_summary(
    mbpo_results,
    batch_size=256,
    real_ratio=0.20,
    updates_per_step=5,
)

## V.3 — Là où MBPO brille : Pendulum, un système que le modèle apprend vite

Changeons de terrain. **Pendulum-v1** n'a que 3 dimensions d'état et une dynamique **lisse, facile à
modéliser** : un petit ensemble la capture en quelques centaines de pas. Le modèle devient donc fiable
**presque immédiatement**, ses rollouts imaginés *aident* au lieu de nuire, et l'avantage
d'efficacité-échantillon de MBPO peut enfin se voir — toujours à budget de pas réels égal.

In [ ]:
# Protocole Pendulum, distinct du budget HalfCheetah.
PD_REAL_STEPS = 10_000
PD_WARMUP = 1_000
SEED_PD = 0
PD_EVAL_EVERY = 1_000


In [ ]:
# Pendulum : le modèle devient fiable vite → MBPO exploite ses rollouts

print("Lancement MBPO (Pendulum)...")
t0 = time.time()
mbpo_pendulum = run_mbpo(
    env_name="Pendulum-v1", total_real_steps=PD_REAL_STEPS, warmup_steps=PD_WARMUP,
    rollout_length=1, rollout_every=150, rollout_batch=300, updates_per_step=15,
    real_ratio=0.1, batch_size=256, model_hidden=64, sac_hidden=128,
    ensemble_size=5, model_fit_steps=60, max_ep_steps=200, seed=SEED_PD,
    eval_every=PD_EVAL_EVERY, eval_episodes=3,
)
print(f"  MBPO terminé en {time.time()-t0:.1f}s")

print("Lancement SAC baseline (Pendulum)...")
t1 = time.time()
sac_pendulum = run_sac_baseline(
    env_name="Pendulum-v1", total_real_steps=PD_REAL_STEPS, warmup_steps=PD_WARMUP,
    sac_hidden=128, batch_size=256, max_ep_steps=200, seed=SEED_PD,
    eval_every=PD_EVAL_EVERY, eval_episodes=3,
)
print(f"  SAC terminé en  {time.time()-t1:.1f}s")

print(f"\nPendulum — retour greedy final  MBPO : {mbpo_pendulum['eval_history'][-1][1]:.1f}")
print(f"Pendulum — retour greedy final  SAC  : {sac_pendulum['eval_history'][-1][1]:.1f}")


In [ ]:
# Pendulum : retour greedy MBPO vs SAC (réutilise extract_curve / extract_eval)
steps_mp, rets_mp = extract_curve(mbpo_pendulum)
steps_sp, rets_sp = extract_curve(sac_pendulum)
ev_mp_x, ev_mp_y = extract_eval(mbpo_pendulum)
ev_sp_x, ev_sp_y = extract_eval(sac_pendulum)

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(steps_mp, rets_mp, alpha=0.20, s=14, color="steelblue")
ax.scatter(steps_sp, rets_sp, alpha=0.20, s=14, color="coral")
ax.plot(ev_mp_x, ev_mp_y, color="steelblue", lw=2.5, marker="o", label="MBPO (greedy)")
ax.plot(ev_sp_x, ev_sp_y, color="coral",     lw=2.5, marker="o", label="SAC (greedy)")
ax.set_xlabel("Pas réels")
ax.set_ylabel("Retour par épisode")
ax.set_title("Pendulum : MBPO vs SAC (retour greedy, budget de pas réels égal)")
ax.legend()
plt.tight_layout()
plt.show()


**Lecture.** Cette fois l'avantage est **net** : à nombre de pas réels égal, la courbe de MBPO monte
**plus vite et plus haut** que celle de SAC. Ce qui change par rapport à HalfCheetah n'est pas
l'algorithme — c'est le **modèle** : sur Pendulum il devient fiable en quelques centaines de pas, donc
les rollouts imaginés deviennent une *vraie* source de données utiles, et l'**UTD élevé** transforme
chaque interaction réelle en dizaines de mises à jour profitables. C'est l'efficacité-échantillon de
MBPO, enfin visible — et la preuve que son moteur n'est pas magique : c'est **un modèle digne de
confiance**.

In [ ]:
# Diagnostics du modèle MBPO sur Pendulum : NLL ↓ = le modèle devient fiable

nll_steps = [s for s, _ in mbpo_pendulum["nll_history"]]
nll_vals  = [v for _, v in mbpo_pendulum["nll_history"]]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))

# NLL du modèle
axes[0].plot(nll_steps, nll_vals, marker="o", color="steelblue", lw=2)
axes[0].set_xlabel("Pas réels")
axes[0].set_ylabel("NLL (espace normalisé)")
axes[0].set_title("Pendulum — NLL de l'ensemble (↓)")

# Retour MBPO brut (tous les épisodes)
steps_m_all = [s for s, _ in mbpo_pendulum["ep_returns"]]
rets_m_all  = [r for _, r in mbpo_pendulum["ep_returns"]]
axes[1].plot(steps_m_all, rets_m_all, color="steelblue", alpha=0.7, lw=1.5)
axes[1].set_xlabel("Pas réels")
axes[1].set_ylabel("Retour par épisode")
axes[1].set_title("Pendulum — retour MBPO (tous les épisodes)")

plt.tight_layout()
plt.show()

**Lecture des diagnostics.** La **NLL du modèle décroît** : l'ensemble apprend la dynamique *et* la
récompense au fil des vraies interactions. C'est cette amélioration du modèle qui rend les rollouts
imaginés de plus en plus dignes de confiance — et qui justifie, dans une vraie run, d'**allonger
progressivement $k$**. Le model-buffer se remplit de transitions fraîches à chaque époque, alimentant
les updates de SAC.

## V.4 — Le bilan : un avantage réel, mais conditionnel

Empilons les leviers, du monde réel vers la politique :

1. **Le modèle prédit dynamique + récompense** → on peut simuler des transitions complètes.
2. **Rollouts courts branchés** → des données imaginées *fiables* (erreur composée maîtrisée).
3. **Deux buffers + petit `real_ratio`** → un flux massif de données, ancré par une pincée de réel.
4. **UTD élevé ($G$ updates/pas)** → chaque vraie interaction entraîne la politique des dizaines de fois.

Mais nos deux essais ont montré la nuance essentielle : **cet avantage est conditionnel à la qualité du
modèle**. Sur Pendulum (modèle vite fiable) MBPO domine SAC à interactions réelles égales ; sur
HalfCheetah à très petite échelle (modèle encore grossier) le résultat reste dans le bruit et l'avantage
ne se voit pas encore. Le modèle n'apprend pas la tâche — il **amplifie** l'apprentissage de SAC, pour
le meilleur quand on peut lui faire confiance, pour le pire sinon. C'est tout le sens de *« when to
trust your model »*.

## V.5 — Voir la politique greedy en vidéo

Au-delà des courbes, on **regarde** ce que la politique apprise donne en mode **déterministe** (greedy) : action $= \tanh(\text{mean})$, sans exploration. On l'évalue d'abord numériquement sur HalfCheetah et Pendulum (MBPO vs SAC), puis on enregistre un court replay `rgb_array` affiché directement dans le notebook.

In [ ]:
# Éval greedy : retour déterministe MBPO vs SAC (à budget de pas réels égal)
for name, env_name, mb, sb, mx in [
    ("HalfCheetah", "HalfCheetah-v5", mbpo_results,  sac_results,  1000),
    ("Pendulum",    "Pendulum-v1",    mbpo_pendulum, sac_pendulum, 200),
]:
    gm, sm = evaluate_greedy(env_name, mb["sac"], episodes=5, max_steps=mx, seed=2_000)
    gs, ss = evaluate_greedy(env_name, sb["sac"], episodes=5, max_steps=mx, seed=2_000)
    print(f"{name:12s} | greedy MBPO : {gm:8.1f} ± {sm:5.1f}   |   greedy SAC : {gs:8.1f} ± {ss:5.1f}")


In [ ]:
# Démonstration finale : replay vidéo notebook, sans fenêtre une fenêtre locale.

In [ ]:
def evaluate_and_display_agent(env_name, sac, label, episodes=2, max_steps=1000, seed=12_345, video_root="videos/12_mbpo"):
    """Évalue une politique SAC greedy et affiche le dernier replay vidéo enregistré."""
    safe_label = label.lower().replace(" ", "_").replace("-", "_")
    video_dir = Path(video_root) / safe_label
    video_dir.mkdir(parents=True, exist_ok=True)

    env = gym.make(env_name, max_episode_steps=max_steps, render_mode="rgb_array")
    env = RecordVideo(
        env,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == episodes - 1,
        name_prefix=f"{safe_label}_greedy",
    )

    low = env.action_space.low.astype(np.float32)
    high = env.action_space.high.astype(np.float32)
    scale = (high - low) / 2.0
    bias = (high + low) / 2.0
    returns = []

    try:
        for ep in range(episodes):
            obs, _ = env.reset(seed=seed + ep)
            ep_ret = 0.0
            steps = 0
            for _ in range(max_steps):
                a_norm = sac.select_action(obs.astype(np.float32), deterministic=True)
                a_env = (np.clip(a_norm, -1.0, 1.0) * scale + bias).astype(np.float32)
                obs, reward, terminated, truncated, _ = env.step(a_env)
                ep_ret += float(reward)
                steps += 1
                if terminated or truncated:
                    break
            returns.append(ep_ret)
            print(f"[{label}] Épisode {ep + 1} : retour={ep_ret:+.1f} ({steps} pas)")
    finally:
        env.close()

    print(f"[{label}] Moyenne : {np.mean(returns):+.1f} +/- {np.std(returns):.1f}")

    videos = sorted(video_dir.glob("*.mp4"), key=lambda path: path.stat().st_mtime)
    if videos:
        print(f"Replay vidéo {label} :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=640))
    else:
        print(f"Aucune vidéo générée pour {label}.")

    return returns


halfcheetah_video_returns = evaluate_and_display_agent(
    "HalfCheetah-v5",
    mbpo_results["sac"],
    "MBPO HalfCheetah",
    episodes=2,
    max_steps=1000,
)
pendulum_video_returns = evaluate_and_display_agent(
    "Pendulum-v1",
    mbpo_pendulum["sac"],
    "MBPO Pendulum",
    episodes=2,
    max_steps=200,
)


**Lecture.** Le retour **greedy** retire le bruit d'exploration : c'est la vraie performance « au test » de chaque politique. À budget de pas réels égal, on compare MBPO et SAC numériquement ; la vidéo sert ensuite à vérifier concrètement le comportement appris, sans ouvrir de fenêtre locale ni bloquer l'exécution du notebook.

---
# Partie VI — Limites, synthèse et suite

## VI.1 — Ce qu'on a construit

Dans ce notebook, on a construit **MBPO** comme une réponse directe à la limite de PETS : au lieu de
planifier par CEM à chaque pas réel, on apprend une **politique SAC** qui agit directement. Le modèle
n'est donc plus utilisé comme contrôleur en ligne, mais comme **générateur de données imaginées**.

Les briques principales sont :

| Brique | Rôle |
|--------|------|
| Ensemble probabiliste | prédire $\Delta s$ et $r$ avec incertitude |
| Buffer réel $\mathcal D_{\text{env}}$ | stocker les vraies transitions |
| Buffer modèle $\mathcal D_{\text{model}}$ | stocker les transitions imaginées courtes |
| Rollouts branchés | démarrer depuis des états réels pour limiter le drift |
| SAC | apprendre une politique off-policy sur un mélange réel + imaginé |
| `real_ratio` | garder une fraction de données réelles pour ancrer les updates |

L'idée centrale tient en une phrase : **on dépense du calcul pour fabriquer beaucoup de transitions
courtes et utiles, puis on laisse SAC apprendre dessus**.

## VI.2 — Ce que MBPO apporte

PETS était très sample-efficient, mais coûteux au moment d'agir : chaque action nécessitait une
optimisation CEM. MBPO déplace ce coût **pendant l'entraînement**. Une fois la politique apprise,
l'action est instantanée : un simple forward pass de l'acteur SAC.

La progression conceptuelle est donc :

$$
\underbrace{\text{PETS}}_{\substack{\text{planification en ligne} \\ \text{CEM à chaque pas}}}
\quad\longrightarrow\quad
\underbrace{\text{MBPO}}_{\substack{\text{modèle pour augmenter le replay} \\ \text{politique SAC rapide}}}
$$

Ce déplacement change le compromis :

| Dimension | PETS | MBPO |
|-----------|------|------|
| Décision au test | CEM/MPC | politique SAC |
| Coût par action | élevé | faible |
| Usage du modèle | planifier des séquences | générer des transitions |
| Horizon imaginé | horizon MPC | rollouts courts |
| Risque principal | plans exploitant le modèle | biais dans le replay imaginé |

MBPO peut se lire comme une version **deep et off-policy de Dyna** : on apprend un modèle, on génère des
expériences fictives, puis on met à jour un agent à partir de vraies et fausses transitions. La
différence est que l'agent n'est plus une table Q, mais SAC.

## VI.3 — Pourquoi les rollouts doivent rester courts

Le résultat important de MBPO n'est pas seulement « utiliser un modèle ». C'est surtout **quand et
combien lui faire confiance**. Un modèle appris est souvent fiable localement, mais ses erreurs se
composent vite :

$$
s_t \to \hat s_{t+1} \to \hat s_{t+2} \to \cdots
$$

À chaque pas imaginé, on nourrit le modèle avec sa propre sortie précédente. Une petite erreur de
dynamique peut donc devenir une trajectoire irréaliste. MBPO évite ce piège en générant des rollouts
**courts**, branchés depuis des états réellement observés. On profite de la généralisation locale du
modèle sans lui demander de rêver trop longtemps.

C'est le sens de la question du papier original : *When to trust your model?* La réponse pratique du
notebook est : **on lui fait confiance près des données, sur quelques pas, puis on revient au réel**.

## VI.4 — Ce que MBPO paie

MBPO n'est pas magique ; il échange une difficulté contre une autre.

| Limite | Conséquence |
|--------|-------------|
| **Biais de modèle** | la politique peut exploiter des transitions imaginées trop optimistes |
| **Coût de calcul** | ensemble probabiliste + nombreux updates SAC = beaucoup de gradient |
| **Réglage sensible** | `rollout_length`, `real_ratio`, UTD et fréquence de refit changent beaucoup le comportement |
| **Pas de planning explicite** | contrairement à PETS, l'agent ne réoptimise pas un plan à chaque état |
| **État vectoriel** | ici on prédit des états compacts ; les pixels exigent un modèle latent |

Les rollouts courts, l'ensemble probabiliste et le mélange réel/imaginé réduisent le biais de modèle,
mais ne l'annulent pas. Si le modèle hallucine des transitions rentables, SAC peut apprendre à les
exploiter. Le diagnostic important n'est donc pas seulement le retour réel : il faut aussi surveiller
la NLL du modèle, la taille du buffer modèle, le schedule de rollout et l'écart entre apprentissage
imaginé et performance réelle.

## VI.5 — Pont vers Dreamer

La dernière limite annonce directement la suite. MBPO imagine dans l'**espace d'état brut** :
HalfCheetah nous donne déjà un vecteur de 17 dimensions contenant les informations utiles. Mais dans
des environnements visuels, prédire le prochain frame pixel par pixel est coûteux, instable, et souvent
inutile pour décider.

**Dreamer** répond en apprenant un **monde latent**. Les observations sont compressées dans un état
latent compact ; la dynamique, la récompense, la valeur et la politique sont ensuite entraînées dans
cet espace imaginé. On ne demande plus au modèle de prédire fidèlement chaque détail de l'observation :
on lui demande de produire un latent utile pour contrôler.

La progression devient :

$$
\underbrace{\text{PETS}}_{\text{planifier avec un modèle d'état}}
\quad\to\quad
\underbrace{\text{MBPO}}_{\text{entraîner SAC avec des rollouts courts}}
\quad\to\quad
\underbrace{\text{Dreamer}}_{\text{apprendre et agir dans un monde latent}}
$$

MBPO pose donc une idée charnière : l'imagination peut améliorer fortement l'efficacité-échantillon,
mais elle doit être **contrôlée**. Dreamer reprend cette idée et la déplace dans un espace latent plus
adapté aux observations riches et aux rollouts plus longs.

## VI.6 — Ce que vous devriez pouvoir réexpliquer

À la fin de ce chapitre, vous devriez pouvoir répondre à ces questions :

1. Pourquoi PETS est sample-efficient mais coûteux au test ?
2. Pourquoi MBPO remplace le planning CEM par une politique SAC ?
3. Pourquoi le modèle prédit $\Delta s$ **et** $r$ ?
4. Pourquoi les rollouts imaginés partent d'états réels ?
5. Pourquoi les rollouts doivent rester courts ?
6. À quoi servent les deux buffers $\mathcal D_{\text{env}}$ et $\mathcal D_{\text{model}}$ ?
7. Pourquoi `real_ratio` empêche le replay imaginé de dériver complètement ?
8. Pourquoi MBPO ressemble à Dyna, mais en version deep off-policy ?
9. Pourquoi Dreamer devient naturel dès que les observations ne sont plus de simples vecteurs ?

> **Lien avec le code du dépôt.** Une version packagée et testée vit dans `src/rl_from_scratch/mbpo/`
> (`dynamics.py`, `sac.py`, `buffer.py`, `agent.py`, `training.py`). Ce notebook réimplémente tout
> *from scratch* pour la pédagogie ; le package, lui, est autonome et se lance via la CLI
> d'entraînement.

## Références

- Janner, M., Fu, J., Zhang, M. & Levine, S. (2019). *When to Trust Your Model: Model-Based Policy Optimization*. NeurIPS. [arXiv:1906.08253](https://arxiv.org/abs/1906.08253).  
  La référence centrale de MBPO : rollouts courts branchés depuis des états réels, mélange réel/imaginé, et analyse du biais de modèle.

- Haarnoja, T., Zhou, A., Abbeel, P. & Levine, S. (2018). *Soft Actor-Critic: Off-Policy Maximum Entropy Deep Reinforcement Learning with a Stochastic Actor*. ICML. [arXiv:1801.01290](https://arxiv.org/abs/1801.01290).  
  L'algorithme de politique utilisé par MBPO : off-policy, replay buffer, twin Q et objectif entropique.

- Chua, K., Calandra, R., McAllister, R. & Levine, S. (2018). *Deep Reinforcement Learning in a Handful of Trials using Probabilistic Dynamics Models*. NeurIPS. [arXiv:1805.12114](https://arxiv.org/abs/1805.12114).  
  PETS, le prédécesseur direct : ensembles probabilistes, incertitude et planification MPC/CEM.

- Sutton, R. S. (1991). *Dyna, an Integrated Architecture for Learning, Planning, and Reacting*. ACM SIGART Bulletin.  
  L'ancêtre conceptuel : apprendre depuis des transitions réelles et simulées dans une même boucle.

- Hafner, D. et al. (2019). *Learning Latent Dynamics for Planning from Pixels*. ICML. [arXiv:1811.04551](https://arxiv.org/abs/1811.04551).  
  PlaNet, étape naturelle vers les modèles latents : planifier dans un espace compact plutôt que dans l'espace d'observation.

- Hafner, D. et al. (2020). *Dream to Control: Learning Behaviors by Latent Imagination*. ICLR. [arXiv:1912.01603](https://arxiv.org/abs/1912.01603).  
  Dreamer pousse l'idée plus loin : entraîner acteur et critique directement sur des trajectoires imaginées dans le latent.